In [ ]:

%pip install -q librosa soundfile transformers torch-geometric pandas seaborn matplotlib
code
#VSC-combined-2
python
# generate_combined_reasoning removed — color_reasoning is canonical
print('generate_combined_reasoning removed; use *_color_reasoning.json/.txt outputs only')

    for i in range(1, len(smooth) - 1):
        if smooth[i] > smooth[i - 1] and smooth[i] >= smooth[i + 1] and smooth[i] > thr:
            t = float(times[i])
            if t - last_t >= min_interval:
                peaks.append(t)
                last_t = t
    if len(peaks) < 2:
        return None
    bpm = (len(peaks) / max(audio_duration_s, 1e-6)) * 60.0
    return float(bpm)


def integrated_gradients_nodes(model, data, target='wheeze', steps=20):
    model.eval()
    x = data.x.detach()
    baseline = torch.zeros_like(x)
    grads_sum = torch.zeros_like(x)
    for alpha in np.linspace(0.0, 1.0, steps):
        x_scaled = (baseline + alpha * (x - baseline)).detach().requires_grad_(True)
        d = Data(x=x_scaled, edge_index=data.edge_index)
        w_logits, c_logits = model(d)
        out = w_logits if target == 'wheeze' else c_logits
        out_scalar = out.view(-1)[0]
        grad = torch.autograd.grad(out_scalar, x_scaled, retain_graph=False, create_graph=False)[0]
        grads_sum += grad
    avg_grads = grads_sum / float(steps)
    attributions = (x - baseline) * avg_grads
    node_attr = attributions.detach().cpu().norm(p=2, dim=1).numpy()
    return node_attr


def grad_times_input_nodes(model, data, target='wheeze'):
    model.eval()
    x = data.x.detach().clone().requires_grad_(True)
    d = Data(x=x, edge_index=data.edge_index)
    w_logits, c_logits = model(d)
    out = w_logits if target == 'wheeze' else c_logits
    out_scalar = out.view(-1)[0]
    grad = torch.autograd.grad(out_scalar, x, retain_graph=False, create_graph=False)[0]
    gxi = (grad * x).detach().cpu().norm(p=2, dim=1).numpy()
    return gxi


def topk_indices(arr, k):
    if arr is None or len(arr) == 0:
        return []
    k = min(int(k), len(arr))
    return [int(i) for i in np.argsort(arr)[-k:][::-1]]


def list_to_csv_value(v):
    if not v:
        return ''
    return ';'.join(str(x) for x in v)


# --- Batch inference per file ---
results = []
for audio_path in files:
    # Load raw audio
    y, _ = librosa.load(str(audio_path), sr=sr, mono=True)
    audio_duration_s = float(len(y) / sr)

    # Frame audio into 0.5s chunks
    frames = []
    for i in range(0, len(y), frame_len):
        f = y[i:i + frame_len]
        if len(f) < frame_len:
            f = np.pad(f, (0, frame_len - len(f)), mode='constant')
        frames.append(f.astype(np.float32))

    if len(frames) == 0:
        print('No frames extracted; skipping', audio_path)
        continue

    # Wav2Vec2 embeddings (batch)
    embeddings = []
    batch_size = 8
    with torch.no_grad():
        for i in range(0, len(frames), batch_size):
            batch = frames[i:i + batch_size]
            inputs = proc(batch, sampling_rate=sr, return_tensors='pt', padding=True)
            out = w2v(**inputs)
            emb = out.last_hidden_state.mean(dim=1).cpu().numpy().astype(np.float32)
            embeddings.append(emb)

    X = np.concatenate(embeddings, axis=0) if len(embeddings) else np.zeros((0, 768), dtype=np.float32)
    if X.shape[0] == 0:
        print('No embeddings computed; skipping', audio_path)
        continue

    # Create graph Data
    edge_index = build_chain_edge_index(X.shape[0])
    data = Data(x=torch.tensor(X, dtype=torch.float32), edge_index=edge_index)

    # Inference (full audio graph)
    start = time.time()
    with torch.no_grad():
        w_logits, c_logits = model(data)

    infer_ms = (time.time() - start) * 1000.0
    w_prob = float(torch.sigmoid(w_logits).view(-1)[0].cpu().item())
    c_prob = float(torch.sigmoid(c_logits).view(-1)[0].cpu().item())

    # Thresholds
    w_thr = 0.5
    c_thr = 0.5
    w_pred = int(w_prob >= w_thr)
    c_pred = int(c_prob >= c_thr)

    # Uncertainty/confidence
    w_unc = None
    c_unc = None
    w_conf = None
    c_conf = None
    if hasattr(model, 'log_var_wheeze'):
        w_logvar = float(model.log_var_wheeze.detach().cpu().item())
        w_unc = float(np.exp(0.5 * w_logvar))
        w_conf = float(1.0 / (1.0 + w_unc))
    if hasattr(model, 'log_var_crackle'):
        c_logvar = float(model.log_var_crackle.detach().cpu().item())
        c_unc = float(np.exp(0.5 * c_logvar))
        c_conf = float(1.0 / (1.0 + c_unc))

    # Breathing rate (full audio)
    breathing_rate_bpm = estimate_breathing_rate_bpm(y, sr, audio_duration_s)

    # Patient state manager (per file)
    state_mgr = PatientStateManager(ema_alpha=0.12, low_delta=0.08, high_delta=0.20, min_samples_for_baseline=5, force_established_after_s=10.0)
    patient_id = audio_path.stem

    # Cumulative windowed reasoning
    window_seconds = 5.0
    frames_per_window = max(1, int(round(window_seconds / frame_seconds)))
    window_rows = []

    for w_start in range(0, X.shape[0], frames_per_window):
        Xw = X[w_start:w_start + frames_per_window]
        if Xw.shape[0] == 0:
            continue
        ew = build_chain_edge_index(Xw.shape[0])
        dw = Data(x=torch.tensor(Xw, dtype=torch.float32), edge_index=ew)
        with torch.no_grad():
            w_l, c_l = model(dw)
        w_p = float(torch.sigmoid(w_l).view(-1)[0].cpu().item())
        c_p = float(torch.sigmoid(c_l).view(-1)[0].cpu().item())
        w_pd = int(w_p >= w_thr)
        c_pd = int(c_p >= c_thr)
        start_sec = float(w_start * frame_seconds)
        end_sec = float(min((w_start + Xw.shape[0]) * frame_seconds, audio_duration_s))

        # Window breathing rate
        start_sample = int(start_sec * sr)
        end_sample = int(end_sec * sr)
        y_window = y[start_sample:end_sample] if end_sample > start_sample else np.array([], dtype=np.float32)
        win_duration = max(end_sec - start_sec, 1e-6)
        breathing_rate_window = estimate_breathing_rate_bpm(y_window, sr, win_duration)

        # Patient state for this window
        state_out = state_mgr.update_and_get_state(patient_id, w_p, c_p, timestamp=start_sec)
        patient_state = state_out.get('overall_state')

        # Optional attributions per window
        ig_topk = []
        gxi_topk = []
        if compute_attributions:
            try:
                ig_scores = integrated_gradients_nodes(model, dw, target=attribution_target, steps=integrated_steps)
                gxi_scores = grad_times_input_nodes(model, dw, target=attribution_target)
                ig_topk = topk_indices(ig_scores, attr_topk)
                gxi_topk = topk_indices(gxi_scores, attr_topk)
            except Exception as e:
                print('Attribution failed for window', start_sec, '->', end_sec, 'error:', e)
                ig_topk = []
                gxi_topk = []

        window_rows.append({
            'start_sec': round(start_sec, 3),
            'end_sec': round(end_sec, 3),
            'num_frames': int(Xw.shape[0]),
            'wheeze_prob': round(w_p, 4),
            'crackle_prob': round(c_p, 4),
            'wheeze_pred': int(w_pd),
            'crackle_pred': int(c_pd),
            'patient_state': patient_state,
            'breathing_rate_bpm': None if breathing_rate_window is None else round(breathing_rate_window, 2),
            'ig_topk': ig_topk,
            'gxi_topk': gxi_topk,
        })

    if len(window_rows) > 0:
        w_win_probs = np.array([r['wheeze_prob'] for r in window_rows], dtype=np.float32)
        c_win_probs = np.array([r['crackle_prob'] for r in window_rows], dtype=np.float32)
        w_win_pred = np.array([r['wheeze_pred'] for r in window_rows], dtype=np.int32)
        c_win_pred = np.array([r['crackle_pred'] for r in window_rows], dtype=np.int32)
        win_summary = {
            'window_seconds': float(window_seconds),
            'frame_seconds': float(frame_seconds),
            'num_windows': int(len(window_rows)),
            'frames_per_window': int(frames_per_window),
            'wheeze_window_positive': int(w_win_pred.sum()),
            'crackle_window_positive': int(c_win_pred.sum()),
            'wheeze_window_ratio': float(w_win_pred.mean()),
            'crackle_window_ratio': float(c_win_pred.mean()),
            'wheeze_prob_mean_window': float(w_win_probs.mean()),
            'crackle_prob_mean_window': float(c_win_probs.mean()),
            'wheeze_prob_max_window': float(w_win_probs.max()),
            'crackle_prob_max_window': float(c_win_probs.max()),
        }
    else:
        win_summary = {
            'window_seconds': float(window_seconds),
            'frame_seconds': float(frame_seconds),
            'num_windows': 0,
            'frames_per_window': int(frames_per_window),
        }

    # Flags
    short_audio = audio_duration_s < frame_seconds
    near_threshold = (abs(w_prob - w_thr) <= 0.02) or (abs(c_prob - c_thr) <= 0.02)

    # Output payload
    request_id = uuid.uuid4().hex
    now_utc = datetime.datetime.now(datetime.timezone.utc)
    timestamp = now_utc.strftime('%Y-%m-%dT%H:%M:%SZ')

    output = {
        'request_id': request_id,
        'timestamp': timestamp,
        'result': {
            'audio_id': audio_path.stem,
            'audio_duration_s': round(audio_duration_s, 3),
            'wheeze': {
                'probability': round(w_prob, 4),
                'prediction': bool(w_pred),
                'confidence': None if w_conf is None else round(w_conf, 4),
            },
            'crackle': {
                'probability': round(c_prob, 4),
                'prediction': bool(c_pred),
                'confidence': None if c_conf is None else round(c_conf, 4),
            },
            'breathing_rate_bpm': None if breathing_rate_bpm is None else round(breathing_rate_bpm, 2),
            'model_version': ckpt.name,
            'inference_time_ms': round(float(infer_ms), 2),
        },
        'reasoning': {
            'thresholds': {'wheeze': w_thr, 'crackle': c_thr},
            'uncertainty_std': {'wheeze': w_unc, 'crackle': c_unc},
            'flags': {
                'short_audio': bool(short_audio),
                'near_threshold': bool(near_threshold),
            },
            'cumulative_windows': win_summary,
        }
    }

    # Save to gat_runs folder
    run_root = repo_root / 'gat_runs'
    run_id = now_utc.strftime('%Y%m%dT%H%M%SZ') + '_' + audio_path.stem + '_' + request_id[:8]
    run_dir = run_root / run_id
    run_dir.mkdir(parents=True, exist_ok=True)

    with open(run_dir / 'result.json', 'w', encoding='utf-8') as f:
        json.dump(output, f, indent=2)

    with open(run_dir / 'reasoning.json', 'w', encoding='utf-8') as f:
        json.dump(output['reasoning'], f, indent=2)

    # Save per-window details (JSON + CSV) for UI/debug
    window_report = {
        'audio_id': audio_path.stem,
        'audio_duration_s': round(audio_duration_s, 3),
        'window_seconds': float(window_seconds),
        'frame_seconds': float(frame_seconds),
        'attributions_enabled': bool(compute_attributions),
        'attribution_target': attribution_target if compute_attributions else None,
        'windows': window_rows,
    }

    with open(run_dir / 'window_report.json', 'w', encoding='utf-8') as f:
        json.dump(window_report, f, indent=2)

    with open(run_dir / 'window_report.csv', 'w', newline='', encoding='utf-8') as f:
        fieldnames = [
            'start_sec', 'end_sec', 'num_frames',
            'wheeze_prob', 'crackle_prob', 'wheeze_pred', 'crackle_pred',
            'patient_state', 'breathing_rate_bpm', 'ig_topk', 'gxi_topk'
        ]
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in window_rows:
            row_csv = dict(row)
            row_csv['ig_topk'] = list_to_csv_value(row_csv.get('ig_topk'))
            row_csv['gxi_topk'] = list_to_csv_value(row_csv.get('gxi_topk'))
            writer.writerow(row_csv)

    output['artifacts'] = {
        'result_json': 'result.json',
        'reasoning_json': 'reasoning.json',
        'window_report_json': 'window_report.json',
        'window_report_csv': 'window_report.csv',
        'run_dir': str(run_dir),
    }

    results.append(output)
    print('Saved outputs to:', run_dir)
    print(json.dumps(output, indent=2))

print('Completed', len(results), 'audio files.')


In [ ]:
# Ensure TRAIN_DEVICE is defined for this notebook
if 'TRAIN_DEVICE' not in globals():
    import torch
    TRAIN_DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print('Set TRAIN_DEVICE ->', TRAIN_DEVICE)


In [ ]:
# Kaggle bootstrap: map dataset paths to this notebook's expected local layout.
import os
import glob
import shutil
from collections import Counter


def _find_icbhi_root_under_kaggle_input():
    # Preferred: explicit ICBHI_final_database folder.
    candidates = sorted(glob.glob('/kaggle/input/**/ICBHI_final_database', recursive=True))
    for path in candidates:
        if os.path.isdir(path) and len(glob.glob(os.path.join(path, '*.wav'))) > 0:
            return path

    # Fallback: infer root from diagnosis location.
    diag_paths = sorted(glob.glob('/kaggle/input/**/ICBHI_Challenge_diagnosis.txt', recursive=True))
    for diag in diag_paths:
        root = os.path.dirname(os.path.dirname(diag))
        if os.path.isdir(root) and len(glob.glob(os.path.join(root, '*.wav'))) > 0:
            return root

    # Last resort: pick directory containing the most wav files.
    wav_paths = glob.glob('/kaggle/input/**/*.wav', recursive=True)
    if len(wav_paths) == 0:
        return None

    counts = Counter(os.path.dirname(w) for w in wav_paths)
    return counts.most_common(1)[0][0]


def _create_symlink_or_copytree(src_dir, dst_dir):
    if os.path.exists(dst_dir):
        return 'exists'

    parent = os.path.dirname(os.path.abspath(dst_dir))
    if parent:
        os.makedirs(parent, exist_ok=True)

# Quick dry training run (self-contained) - inlines improved model + trainer
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
try:
    from torch_geometric.nn import GATConv, SAGEConv, BatchNorm
except Exception as e:
    print('Missing torch_geometric components:', e)
    raise

# ImprovedRespiratoryGAT (inlined)
class ImprovedRespiratoryGAT(nn.Module):
    def __init__(self, input_dim=768, hidden_dim=256, num_layers=4, num_heads=4, dropout=0.4):
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.gat_layers = nn.ModuleList()
        self.norms = nn.ModuleList()
        for i in range(num_layers):
            if i % 2 == 0:
                conv = GATConv(hidden_dim, max(1, hidden_dim // num_heads), heads=num_heads, concat=True, dropout=dropout)
            else:
                conv = SAGEConv(hidden_dim, hidden_dim)
            self.gat_layers.append(conv)
            self.norms.append(BatchNorm(hidden_dim))
        self.attention_pool = nn.Sequential(
            nn.Linear(hidden_dim, max(4, hidden_dim // 4)),
            nn.Tanh(),
            nn.Linear(max(4, hidden_dim // 4), 1),
        )
        self.wheeze_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1),
        )
        self.crackle_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1),
        )
        self.log_var_wheeze = nn.Parameter(torch.tensor(0.0))
        self.log_var_crackle = nn.Parameter(torch.tensor(0.0))
        self.dropout = dropout

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        batch = data.batch if hasattr(data, 'batch') else None
        x = self.input_proj(x)
        residuals = []
        for i, (conv, norm) in enumerate(zip(self.gat_layers, self.norms)):
            x_new = conv(x, edge_index)
            x_new = F.elu(x_new)
            x_new = norm(x_new)
            if i > 0 and i % 2 == 0:
                x_new = x_new + residuals[-1]
            x = F.dropout(x_new, p=self.dropout, training=self.training)
            residuals.append(x)
        if batch is not None:
            attn_scores = self.attention_pool(x).squeeze(-1)
            x_graph = []
            for b in torch.unique(batch):
                mask = batch == b
                scores = attn_scores[mask]
                weights = torch.softmax(scores, dim=0).unsqueeze(-1)
                x_graph.append((x[mask] * weights).sum(dim=0))
            x = torch.stack(x_graph, dim=0)
        else:
            attn_scores = self.attention_pool(x).squeeze(-1)
            weights = torch.softmax(attn_scores, dim=0).unsqueeze(-1)
            x = (x * weights).sum(dim=0, keepdim=True)
        w_logits = self.wheeze_head(x).squeeze(-1)
        c_logits = self.crackle_head(x).squeeze(-1)
        return w_logits, c_logits

# Postprocessing / calibration (inlined)
class PredictionCalibrator:
    def __init__(self, config=None):
        self.config = config or {}
        self.history = {'wheeze': [], 'crackle': []}
    def temporal_smoothing(self, predictions, window=3):
        predictions = np.asarray(predictions)
        if len(predictions) < window:
            return predictions
        smoothed = []
        half = window // 2
        for i in range(len(predictions)):
            start = max(0, i - half)
            end = min(len(predictions), i + half + 1)
            smoothed.append(np.mean(predictions[start:end]))
        return np.array(smoothed)
    def adaptive_threshold(self, predictions, base_threshold, recency_weight=0.3):
        if len(self.history['wheeze']) < 10:
            return base_threshold
        recent_fp = np.mean([1 for p in self.history['wheeze'][-10:] if p > base_threshold and p < 0.5])
        adjustment = recency_weight * recent_fp
        adjusted = min(base_threshold + adjustment, 0.85)
        return adjusted
    def minimum_duration(self, binary_predictions, min_frames=3):
        if len(binary_predictions) < min_frames:
            return np.array(binary_predictions)
        result = np.array(binary_predictions).copy()
        in_detection = False
        start_idx = 0
        for i, pred in enumerate(binary_predictions):
            if pred == 1 and not in_detection:
                in_detection = True
                start_idx = i
            elif pred == 0 and in_detection:
                in_detection = False
                if i - start_idx < min_frames:
                    result[start_idx:i] = 0
        if in_detection and len(binary_predictions) - start_idx < min_frames:
            result[start_idx:] = 0
        return result

# Focal loss + trainer (inlined)
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    def forward(self, inputs, targets):
        bce_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        pt = torch.exp(-bce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * bce_loss
        return focal_loss.mean()

def evaluate_with_postprocessing(model, loader, device, threshold=0.65, calibrator=None):
    model.eval()
    w_probs, c_probs, w_trues, c_trues = [], [], [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            w_logits, c_logits = model(batch)
            w_prob = torch.sigmoid(w_logits).cpu().numpy()
            c_prob = torch.sigmoid(c_logits).cpu().numpy()
            w_probs.extend(w_prob.tolist())
            c_probs.extend(c_prob.tolist())
            if hasattr(batch, 'y_wheeze'):
                w_trues.extend(batch.y_wheeze.cpu().numpy().tolist())
            else:
                w_trues.extend([0] * len(w_prob))
            if hasattr(batch, 'y_crackle'):
                c_trues.extend(batch.y_crackle.cpu().numpy().tolist())
            else:
                c_trues.extend([0] * len(c_prob))
    w_probs = np.array(w_probs)
    c_probs = np.array(c_probs)
    w_trues = np.array(w_trues)
    c_trues = np.array(c_trues)
    if calibrator is not None:
        w_probs = calibrator.temporal_smoothing(w_probs)
        c_probs = calibrator.temporal_smoothing(c_probs)
    w_pred = (w_probs >= threshold).astype(int)
    c_pred = (c_probs >= threshold).astype(int)
    try:
        from sklearn.metrics import f1_score
    except Exception:
        f1_score = None
    if f1_score is None:
        return {'wheeze_f1': float((w_pred == w_trues).mean()), 'crackle_f1': float((c_pred == c_trues).mean()), 'weighted_f1': float(((w_pred == w_trues).mean() + (c_pred == c_trues).mean()) / 2.0)}
    w_f1 = f1_score(w_trues, w_pred, average='weighted')
    c_f1 = f1_score(c_trues, c_pred, average='weighted')
    return {'wheeze_f1': float(w_f1), 'crackle_f1': float(c_f1), 'weighted_f1': float((w_f1 + c_f1) / 2.0)}

def train_improved_model(model, train_loader, val_loader, device, epochs=60, initial_lr=0.0005, warmup_epochs=5, target_f1=0.70):
    optimizer = torch.optim.AdamW(model.parameters(), lr=initial_lr, weight_decay=0.0001)
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return (epoch + 1) / max(1, warmup_epochs)
        else:
            progress = (epoch - warmup_epochs) / max(1, (epochs - warmup_epochs))
            return 0.5 * (1 + np.cos(np.pi * progress))
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    focal_loss = FocalLoss(alpha=0.25, gamma=2.0)
    best_val_f1 = 0.0
    patience_counter = 0
    history = []
    calibrator = PredictionCalibrator(None)
    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        for batch in train_loader:
            batch = batch.to(device)
            w_logits, c_logits = model(batch)
            loss_w = focal_loss(w_logits, batch.y_wheeze.float())
            loss_c = focal_loss(c_logits, batch.y_crackle.float())
            loss = 1.25 * loss_w + loss_c
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += float(loss.item())
        scheduler.step()
        avg_loss = total_loss / max(1, len(train_loader))
        val_metrics = evaluate_with_postprocessing(model, val_loader, device, threshold=0.65, calibrator=calibrator)
        val_f1 = val_metrics.get('weighted_f1', 0.0)
        history.append({'epoch': epoch, 'train_loss': avg_loss, 'val_weighted_f1': val_f1, 'learning_rate': scheduler.get_last_lr()[0]})
        print(f'Epoch {epoch:3d}/{epochs} | Loss: {avg_loss:.4f} | Val Weighted F1: {val_f1:.4f} | LR: {scheduler.get_last_lr()[0]:.6f}')
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            torch.save(model.state_dict(), 'best_improved_respiratory_gat.pt')
            patience_counter = 0
            print(f'  ✓ New best model! Weighted F1: {best_val_f1:.4f}')
            if best_val_f1 >= target_f1:
                print(f'TARGET REACHED at epoch {epoch}! (Weighted F1 >= {target_f1})')
                break
        else:
            patience_counter += 1
            if patience_counter >= 15:
                print(f'Early stopping at epoch {epoch}')
                break
    return model, history

# Synthetic dataset with graph-level scalar labels
class SyntheticRespiratoryDataset(torch.utils.data.Dataset):
    def __init__(self, n_samples=40, frames=8, feat_dim=768, pos_prob_w=0.10, pos_prob_c=0.15):
        self.n = int(n_samples)
        self.frames = int(frames)
        self.feat_dim = int(feat_dim)
        self.pos_prob_w = float(pos_prob_w)
        self.pos_prob_c = float(pos_prob_c)
    def __len__(self):
        return self.n
    def __getitem__(self, idx):
        x = torch.randn(self.frames, self.feat_dim, dtype=torch.float32)
        edges = []
        for i in range(self.frames - 1):
            edges.append([i, i + 1])
            edges.append([i + 1, i])
        edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous() if len(edges) else torch.empty((2, 0), dtype=torch.long)
        y_w = torch.bernoulli(torch.tensor(self.pos_prob_w, dtype=torch.float32)).float()
        y_c = torch.bernoulli(torch.tensor(self.pos_prob_c, dtype=torch.float32)).float()
        return Data(x=x, edge_index=edge_index, y_wheeze=y_w, y_crackle=y_c)

# Build tiny datasets and loaders
train_ds = SyntheticRespiratoryDataset(n_samples=40, frames=8)
val_ds = SyntheticRespiratoryDataset(n_samples=12, frames=8)
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=6, shuffle=False)

# Device from earlier notebook or fallback
device = TRAIN_DEVICE if 'TRAIN_DEVICE' in globals() else torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

# Small model for smoke test
model = ImprovedRespiratoryGAT(input_dim=768, hidden_dim=128, num_layers=2, num_heads=4, dropout=0.2).to(device)

# Run short dry training (2 epochs)
model, history = train_improved_model(model, train_loader, val_loader, device, epochs=2, initial_lr=5e-4, warmup_epochs=1, target_f1=0.80)
print('Dry run finished. Last history entry:')
print(history[-1] if len(history) else history)

IN_KAGGLE = os.path.exists('/kaggle/input')


def resolve_audio_folder(path_hint):
    candidates = []

    if path_hint:
        candidates.append(path_hint)

    if IN_KAGGLE:
        candidates.extend([
            '/kaggle/input/datasets/catherinenassali/icbhi-final-database/ICBHI_final_database',
            '/kaggle/input/datasets/catherinenassali/icbhi-final-database',
        ])
        candidates.extend(sorted(glob.glob('/kaggle/input/**/ICBHI_final_database', recursive=True)))

    for cand in candidates:
        if os.path.isdir(cand):
            if len(glob.glob(os.path.join(cand, '*.wav'))) > 0:
                return cand
            subdirs = [d for d in glob.glob(os.path.join(cand, '*')) if os.path.isdir(d)]
            for s in subdirs:
                if len(glob.glob(os.path.join(s, '*.wav'))) > 0:
                    return s

    raise FileNotFoundError('Could not resolve an audio folder containing .wav files.')


def resolve_diagnosis_file(path_hint):
    candidates = []

    if path_hint:
        candidates.append(path_hint)

    if path_hint and os.path.isdir(path_hint):
        candidates.extend([
            os.path.join(path_hint, 'ICBHI_Challenge_diagnosis.txt'),
            os.path.join(path_hint, 'important', 'ICBHI_Challenge_diagnosis.txt'),
        ])
        candidates.extend(sorted(glob.glob(os.path.join(path_hint, '**', 'ICBHI_Challenge_diagnosis.txt'), recursive=True)))

    candidates.extend([
        'ICBHI_final_database/important/ICBHI_Challenge_diagnosis.txt',
        'ICBHI_Challenge_diagnosis.txt',
    ])

    if IN_KAGGLE:
        candidates.extend(sorted(glob.glob('/kaggle/input/**/ICBHI_Challenge_diagnosis.txt', recursive=True)))

    for cand in candidates:
        if os.path.isfile(cand):
            return cand

    raise FileNotFoundError('Could not resolve ICBHI_Challenge_diagnosis.txt from provided path hint.')


if IN_KAGGLE:
    audio_folder_hint = '/kaggle/input/datasets/catherinenassali/icbhi-final-database/ICBHI_final_database'
    diagnosis_file_hint = '/kaggle/input/datasets/catherinenassali/important'
else:
    # Prefer the absolute path if the user set `audio_folder_path` earlier in the notebook.
    audio_folder_hint = globals().get('audio_folder_path', 'ICBHI_final_database')
    diagnosis_file_hint = globals().get('diagnosis_file_path', 'ICBHI_final_database/important/ICBHI_Challenge_diagnosis.txt')

audio_folder_path = resolve_audio_folder(audio_folder_hint)
diagnosis_file_path = resolve_diagnosis_file(diagnosis_file_hint)

print('Audio folder path:', audio_folder_path)
print('Audio folder exists:', os.path.exists(audio_folder_path))
print('Diagnosis file path:', diagnosis_file_path)
print('Diagnosis file exists:', os.path.exists(diagnosis_file_path))


def load_diagnosis_file(path):
    if os.path.isdir(path):
        raise IsADirectoryError('Diagnosis path is a directory. Expected .txt file: ' + str(path))
    if not os.path.isfile(path):
        raise FileNotFoundError('Diagnosis file not found: ' + str(path))

    df = pd.read_csv(path, sep='\t', header=None, names=['patient_id', 'diagnosis'])
    df['patient_id'] = df['patient_id'].astype(str)
    return df


def scan_dataset(audio_folder):
    wav_files = sorted(glob.glob(os.path.join(audio_folder, '*.wav')))
    ann_files = sorted(glob.glob(os.path.join(audio_folder, '*.txt')))

    wav_map = {os.path.splitext(os.path.basename(f))[0]: f for f in wav_files}
    ann_map = {os.path.splitext(os.path.basename(f))[0]: f for f in ann_files}

    paired = []
    missing_ann = 0
    for base, wav_path in wav_map.items():
        if base in ann_map:
            paired.append((base, wav_path, ann_map[base]))
        else:
            missing_ann += 1

    return paired, missing_ann, len(wav_files)


def extract_patient_id(base_filename):
    return base_filename.split('_')[0]


def build_metadata_table(paired_files, diagnosis_df):
    rows = []
    diag_map = dict(zip(diagnosis_df['patient_id'], diagnosis_df['diagnosis']))

    for base, wav_path, ann_path in paired_files:
        patient_id = extract_patient_id(base)
        diagnosis = diag_map.get(patient_id, 'Unknown')
        rows.append({
            'file_id': base,
            'patient_id': patient_id,
            'diagnosis': diagnosis,
            'wav_path': wav_path,
            'ann_path': ann_path,
        })

    return pd.DataFrame(rows)


diagnosis_df = load_diagnosis_file(diagnosis_file_path)
paired_files, missing_ann, total_wavs = scan_dataset(audio_folder_path)
meta_df = build_metadata_table(paired_files, diagnosis_df)

print('\n=== Dataset Summary ===')
print('Diagnosis table shape:', diagnosis_df.shape)
print('Total wav files:', total_wavs)
print('Paired wav+annotation:', len(paired_files))
print('Missing annotation files:', missing_ann)
print('Metadata shape:', meta_df.shape)
print('\nExample pair:')
print(paired_files[0] if len(paired_files) else 'No pairs found')
print('\nDiagnosis distribution:')
print(meta_df['diagnosis'].value_counts())

In [ ]:
# Kaggle GPU-compat patch: safe wav2vec CPU-fallback and safe cache compute
# Inserts CPU-fallback wrappers so wav2vec inference won't crash on incompatible CUDA kernels.
import os
import json
import pickle
from pathlib import Path

import librosa
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv
from transformers import Wav2Vec2Model, Wav2Vec2Processor


def _safe_model_forward(model, input_values):
    try:
        with torch.no_grad():
            return model(input_values)
    except Exception as e:
        print("Wav2Vec forward on device failed, falling back to CPU:", repr(e))
        model_cpu = model.to("cpu")
        out = model_cpu(input_values.cpu())
        try:
            model.to(torch.device("cuda"))
        except Exception:
            pass
        return out


def wav2vec_frame_embedding_safe(frame_audio, sr=16000):
    proc = globals().get("processor", None)
    model = globals().get("wav2vec_model", None)
    if proc is None or model is None:
        proc = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
        model = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base-960h")

    inputs = proc(frame_audio, sampling_rate=sr, return_tensors="pt", padding=True)
    input_values = inputs.input_values

    try:
        model_device = next(model.parameters()).device
    except StopIteration:
        model_device = torch.device("cpu")

    if model_device.type == "cuda" and torch.cuda.is_available():
        input_values = input_values.to(model_device)
    else:
        input_values = input_values.cpu()

    out = _safe_model_forward(model, input_values)
    hidden_states = out.last_hidden_state
    emb = hidden_states.mean(dim=1).squeeze(0)
    return emb.cpu().numpy().astype("float32")


# Monkeypatch Wav2Vec2FeatureCache.compute_and_cache_file for CPU fallback (if class exists)
if "Wav2Vec2FeatureCache" in globals():

    def _safe_compute_and_cache_file(self, audio_path, batch_size=32):
        cache_path = self._cache_key(audio_path)
        if os.path.exists(cache_path):
            with open(cache_path, "rb") as f:
                return pickle.load(f)

        y, _ = librosa.load(audio_path, sr=self.sr, mono=True)
        num_frames = len(y) // self.frame_samples
        frames = [
            y[i * self.frame_samples : (i + 1) * self.frame_samples]
            for i in range(num_frames)
        ]

        embeddings = []
        try:
            with torch.no_grad():
                for i in range(0, len(frames), batch_size):
                    batch = frames[i : i + batch_size]
                    inputs = self.processor(
                        batch,
                        sampling_rate=self.sr,
                        return_tensors="pt",
                        padding=True,
                    )
                    input_values = inputs.input_values.to(self._device)
                    out = self.model(input_values)
                    emb = out.last_hidden_state.mean(dim=1).cpu().numpy()
                    embeddings.append(emb)
        except Exception as e:
            print(
                "Wav2Vec2 cache compute failed on device; falling back to CPU:",
                repr(e),
            )
            model_cpu = self.model.to("cpu")
            for i in range(0, len(frames), batch_size):
                batch = frames[i : i + batch_size]
                inputs = self.processor(
                    batch,
                    sampling_rate=self.sr,
                    return_tensors="pt",
                    padding=True,
                )
                input_values = inputs.input_values.cpu()
                out = model_cpu(input_values)
                emb = out.last_hidden_state.mean(dim=1).cpu().numpy()
                embeddings.append(emb)
            try:
                self.model.to(self._device)
            except Exception:
                pass

        embeddings = (
            np.concatenate(embeddings, axis=0)
            if len(embeddings)
            else np.zeros((0, 768), dtype=np.float32)
        )
        data = {
            "embeddings": embeddings.astype(np.float32),
            "audio_duration": len(y) / self.sr,
            "num_frames": num_frames,
        }
        with open(cache_path, "wb") as f:
            pickle.dump(data, f, protocol=pickle.HIGHEST_PROTOCOL)
        return data

    Wav2Vec2FeatureCache.compute_and_cache_file = _safe_compute_and_cache_file
    print("Patched Wav2Vec2FeatureCache.compute_and_cache_file for CPU fallback.")


# Override global wav2vec_frame_embedding used elsewhere
globals()["wav2vec_frame_embedding"] = wav2vec_frame_embedding_safe
print("Inserted safe wav2vec wrapper (CPU fallback).")



In [ ]:
# Inspect one batch shapes for debugging
import torch

device = globals().get('TRAIN_DEVICE', torch.device('cuda' if torch.cuda.is_available() else 'cpu'))
print('Using device ->', device)

batch_sample = next(iter(train_loader))
print('Keys:', list(batch_sample.keys()))
if hasattr(batch_sample, 'y_wheeze'):
    print('y_wheeze.shape ->', batch_sample.y_wheeze.shape)
if hasattr(batch_sample, 'y_crackle'):
    print('y_crackle.shape ->', batch_sample.y_crackle.shape)
if hasattr(batch_sample, 'batch'):
    print('batch.vector shape ->', getattr(batch_sample, 'batch').shape)
print('x.shape ->', batch_sample.x.shape)

batch_sample = batch_sample.to(device)
w_logits, c_logits = model(batch_sample)
print('w_logits.shape:', getattr(w_logits, 'shape', None))
print('c_logits.shape:', getattr(c_logits, 'shape', None))


In [ ]:
# Re-load improved modules robustly (search project root for `model_comparisons`)
import importlib.util, os, types, sys, pathlib


def _find_project_root(start=None, marker='model_comparisons', required_files=None, max_up=8):
    required_files = required_files or ['gnn_improved_model.py', 'train_improved.py']
    p = pathlib.Path(start or '.').resolve()
    if p.is_file():
        p = p.parent
    for _ in range(max_up + 1):
        candidate = p / marker
        if candidate.exists() and candidate.is_dir():
            # prefer marker dir that contains required files
            has_all = all((candidate / rf).exists() for rf in required_files)
            if has_all:
                return str(p)
        if p.parent == p:
            break
        p = p.parent
    # fallback: try top-level parent of start
    return str(pathlib.Path(start or '.').resolve().parent)

# Prefer an existing `proj_root` if it already points at the repo root
if 'proj_root' in globals() and os.path.isdir(proj_root) and os.path.isdir(os.path.join(proj_root, 'model_comparisons')) and os.path.isfile(os.path.join(proj_root, 'model_comparisons', 'gnn_improved_model.py')):
    _proj_root = pathlib.Path(proj_root)
else:
    # Prefer nb_dir when available (notebook directory or file), else current working dir
    start = pathlib.Path(nb_dir) if 'nb_dir' in globals() else pathlib.Path().resolve()
    _proj_root = pathlib.Path(_find_project_root(start))

proj_root = str(_proj_root)
print('Using proj_root ->', proj_root)

model_file = os.path.join(proj_root, 'model_comparisons', 'gnn_improved_model.py')
train_file = os.path.join(proj_root, 'model_comparisons', 'train_improved.py')

if not os.path.isfile(model_file):
    raise FileNotFoundError(f"Model file not found: {model_file}")

spec = importlib.util.spec_from_file_location('gnn_improved_model_live', model_file)
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)

# Ensure package namespace exists so sibling imports inside train_improved work
pkg = types.ModuleType('model_comparisons')
pkg.__path__ = [os.path.join(proj_root, 'model_comparisons')]
sys.modules['model_comparisons'] = pkg
sys.modules['model_comparisons.gnn_improved_model'] = mod

GATv2RespiratoryGAT = getattr(mod, 'GATv2RespiratoryGAT')
ImprovedRespiratoryGAT = getattr(mod, 'ImprovedRespiratoryGAT')

spec2 = importlib.util.spec_from_file_location('train_improved_live', train_file)
mod2 = importlib.util.module_from_spec(spec2)
spec2.loader.exec_module(mod2)
train_improved_model = getattr(mod2, 'train_improved_model')

print('Re-loaded updated modules')


In [ ]:
# Start full training (30 epochs)
import time, traceback, importlib.util, os, sys
print('Starting full training (30 epochs) ...')
start_time = time.time()

try:
    # Ensure train_improved_model available
    try:
        from model_comparisons.train_improved import train_improved_model
    except Exception:
        spec = importlib.util.spec_from_file_location(
            'train_improved_live', os.path.join(proj_root, 'model_comparisons', 'train_improved.py')
        )
        mod_train = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(mod_train)
        train_improved_model = getattr(mod_train, 'train_improved_model')

    # Ensure model classes available
    try:
        from model_comparisons.gnn_improved_model import GATv2RespiratoryGAT, ImprovedRespiratoryGAT
    except Exception:
        mod_gnn = sys.modules.get('model_comparisons.gnn_improved_model')
        if mod_gnn is None:
            spec2 = importlib.util.spec_from_file_location(
                'gnn_improved_model_live', os.path.join(proj_root, 'model_comparisons', 'gnn_improved_model.py')
            )
            mod_gnn = importlib.util.module_from_spec(spec2)
            spec2.loader.exec_module(mod_gnn)
            sys.modules['model_comparisons.gnn_improved_model'] = mod_gnn
        GATv2RespiratoryGAT = getattr(mod_gnn, 'GATv2RespiratoryGAT')
        ImprovedRespiratoryGAT = getattr(mod_gnn, 'ImprovedRespiratoryGAT')

    # instantiate model (prefer GATv2)
    try:
        model_full = GATv2RespiratoryGAT(input_dim=768, hidden_dim=256, num_layers=4, num_heads=4, dropout=0.2).to(TRAIN_DEVICE)
        print('Using GATv2RespiratoryGAT')
    except Exception as e:
        print('GATv2 instantiation failed, falling back to ImprovedRespiratoryGAT:', e)
        model_full = ImprovedRespiratoryGAT(input_dim=768, hidden_dim=256, num_layers=4, num_heads=4, dropout=0.2).to(TRAIN_DEVICE)

    epochs = 30
    save_path = 'best_gatv2_full.pt'

    model_full, history_full = train_improved_model(
        model=model_full,
        train_loader=train_loader,
        val_loader=val_loader,
        device=TRAIN_DEVICE,
        epochs=epochs,
        initial_lr=5e-4,
        warmup_epochs=1,
        target_f1=0.65,
        config=None,
        attention_l2=1e-5,
        use_uncertainty=True,
        wheeze_pos_weight=w_pw,
        crackle_pos_weight=c_pw,
        use_focal=False,
        focal_gamma=2.0,
        save_path=save_path,
    )

    print('Full training complete. Saved:', save_path)
    print('History entries:', len(history_full))

except Exception as e:
    traceback.print_exc()
    print('Training failed:', e)

finally:
    print('Elapsed (s):', round(time.time() - start_time, 1))


In [ ]:
# Quick shape test: run GATv2 forward on one batch
batch_sample = next(iter(train_loader))
print('batch sizes -> nodes:', batch_sample.x.shape[0], 'graphs:', batch_sample.y_wheeze.shape[0])

batch_sample = batch_sample.to(TRAIN_DEVICE)
try:
    test_model = GATv2RespiratoryGAT(input_dim=768, hidden_dim=256, num_layers=4, num_heads=4, dropout=0.2).to(TRAIN_DEVICE)
    w_t, c_t = test_model(batch_sample)
    print('GATv2RespiratoryGAT outputs -> w:', getattr(w_t,'shape',None), 'c:', getattr(c_t,'shape',None))
except Exception as e:
    import traceback; traceback.print_exc()
    print('GATv2 forward failed:', e)

try:
    test_model2 = ImprovedRespiratoryGAT(input_dim=768, hidden_dim=256, num_layers=4, num_heads=4, dropout=0.2).to(TRAIN_DEVICE)
    w2, c2 = test_model2(batch_sample)
    print('ImprovedRespiratoryGAT outputs -> w:', getattr(w2,'shape',None), 'c:', getattr(c2,'shape',None))
except Exception as e:
    import traceback; traceback.print_exc()
    print('ImprovedRespiratoryGAT forward failed:', e)


In [ ]:
# Debug: inspect torch.unique(batch) and manual attention pooling for GATv2
batch = next(iter(train_loader))
print('raw nodes:', batch.x.shape[0], 'graphs:', batch.y_wheeze.shape[0])

batch = batch.to(TRAIN_DEVICE)
print('batch vector shape:', batch.batch.shape, 'dtype:', batch.batch.dtype)
unique = torch.unique(batch.batch)
print('unique values:', unique, 'count:', unique.numel())

# instantiate small GATv2 for debug (reuse test_model if present)
try:
    gm = GATv2RespiratoryGAT(input_dim=768, hidden_dim=256, num_layers=4, num_heads=4, dropout=0.2).to(TRAIN_DEVICE)
except Exception:
    gm = model_full

x = gm.input_proj(batch.x)
attn_scores = gm.attention_pool(x).squeeze(-1)
print('x.shape after proj:', x.shape, 'attn_scores.shape:', attn_scores.shape)

x_graph = []
for b in unique:
    mask = (batch.batch == b)
    print('b', int(b.item()), 'mask_sum', int(mask.sum().item()))
    scores = attn_scores[mask]
    weights = torch.softmax(scores, dim=0).unsqueeze(-1)
    pooled = (x[mask] * weights).sum(dim=0)
    print(' pooled shape:', pooled.shape)
    x_graph.append(pooled)

x_graph_stacked = torch.stack(x_graph, dim=0)
print('x_graph_stacked.shape:', x_graph_stacked.shape)


In [ ]:
# Full training: ImprovedRespiratoryGAT (30 epochs)
print("Starting ImprovedRespiratoryGAT full training (30 epochs)...")

from model_comparisons.train_improved import train_improved_model
from model_comparisons.gnn_improved_model import ImprovedRespiratoryGAT

save_path = "best_improved_30epochs.pt"

# Ensure we have a usable ImprovedRespiratoryGAT instance
try:
    is_ok = isinstance(model, ImprovedRespiratoryGAT)
except Exception:
    is_ok = False

if not is_ok:
    print('`model` is not an ImprovedRespiratoryGAT instance; creating a new one')
    model = ImprovedRespiratoryGAT(input_dim=768, hidden_dim=256, num_layers=4, num_heads=4, dropout=0.4)

model = model.to(TRAIN_DEVICE)

model_trained, history_trained = train_improved_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=TRAIN_DEVICE,
    epochs=30,
    save_path=save_path,
)

print('Training finished, saved:', save_path)
trained_model = model_trained
training_history = history_trained


In [ ]:
# Re-run full training: force 30 epochs (disable early stopping)
print("Re-running ImprovedRespiratoryGAT training for full 30 epochs (no early stop)...")

from model_comparisons.train_improved import train_improved_model
from model_comparisons.gnn_improved_model import ImprovedRespiratoryGAT

save_path = "best_improved_30epochs_no_earlystop.pt"

# Create a fresh model to avoid any checkpoint/optimizer state
model = ImprovedRespiratoryGAT(input_dim=768, hidden_dim=256, num_layers=4, num_heads=4, dropout=0.4)
model = model.to(TRAIN_DEVICE)

config = {"training": {"epochs": 30, "learning_rate": 0.0005, "warmup_epochs": 5}}

model_trained, history_trained = train_improved_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=TRAIN_DEVICE,
    epochs=30,
    config=config,
    target_f1=999.0,  # effectively disable early stopping via target
    save_path=save_path,
)

print('Training finished, saved:', save_path)
trained_model_full = model_trained
training_history_full = history_trained


# Tests and evaluations

In [ ]:
# Deployment test: load checkpoint, evaluate on validation set, and run single-graph inference using web_deploy.deploy_utils
import os
import numpy as np
import torch
from model_comparisons.train_improved import evaluate_with_postprocessing

try:
    import web_deploy.deploy_utils as deploy_utils
except Exception:
    import sys, pathlib
    repo_root = pathlib.Path().resolve()
    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))
    import web_deploy.deploy_utils as deploy_utils

# locate checkpoint (prefer the fully-trained saved file)
candidates = [
    "best_improved_30epochs_no_earlystop.pt",
    "best_improved_30epochs.pt",
    "best_improved_30epochs_no_earlystop.pt",
]
ckpt = next((p for p in candidates if os.path.exists(p)), None)
if ckpt is None:
    print("No checkpoint found. Please run training or set `ckpt` path.")
else:
    print("Using checkpoint:", ckpt)
    dm = deploy_utils.load_deployment_model(ckpt, device="cpu")
    print("Loaded deployed model; sample param device:", next(dm.parameters()).device)

    # Evaluate on the validation set using the model loaded to CPU
    metrics = evaluate_with_postprocessing(dm, val_loader, device="cpu", threshold=0.5)
    print("Validation metrics (threshold=0.5):", metrics)

    # Single-graph test: use the first graph from a validation batch
    batch = next(iter(val_loader))
    if hasattr(batch, "batch"):
        mask = (batch.batch == 0)
        node_idx = mask.nonzero(as_tuple=False).squeeze(-1).cpu().numpy()
        if node_idx.size == 0:
            print("Empty first graph; skipping single-graph test.")
        else:
            edge_index = batch.edge_index.cpu().numpy()
            cols_mask = np.isin(edge_index[0], node_idx) & np.isin(edge_index[1], node_idx)
            sub_edges = edge_index[:, cols_mask].copy()
            idx_map = {old: int(i) for i, old in enumerate(node_idx)}
            reindexed = np.vectorize(idx_map.get)(sub_edges)
            x = batch.x[mask].cpu().numpy()

            # Minimal object expected by the model's forward
            class D:
                pass

            d = D()
            d.x = torch.tensor(x, dtype=torch.float32)
            d.edge_index = torch.tensor(reindexed, dtype=torch.long)

            with torch.no_grad():
                w_logits, c_logits = dm(d)
            w_direct = torch.sigmoid(w_logits).cpu().numpy()
            c_direct = torch.sigmoid(c_logits).cpu().numpy()

            w2, c2 = deploy_utils.predict_from_arrays(dm, x, reindexed, device="cpu")

            print("Direct probs:", w_direct, c_direct)
            print("Deploy utils probs:", w2, c2)
            print("abs diffs:", float(abs(w_direct - w2)), float(abs(c_direct - c2)))
    else:
        print("Batch not batched; invoking model on batch directly.")
        with torch.no_grad():
            w_logits, c_logits = dm(batch)
        print("probs:", torch.sigmoid(w_logits).cpu().numpy(), torch.sigmoid(c_logits).cpu().numpy())


In [ ]:
# Ensure a minimal `val_loader` is present for deployment tests
if 'val_loader' not in globals():
    print('val_loader missing — creating a small synthetic validation loader for deployment testing')
    import torch
    from torch_geometric.data import Data
    from torch_geometric.loader import DataLoader

    class SmallDataset(torch.utils.data.Dataset):
        def __init__(self, n=6, frames=8, feat_dim=768):
            self.n = n
            self.frames = frames
            self.feat_dim = feat_dim

        def __len__(self):
            return self.n

        def __getitem__(self, idx):
            x = torch.randn(self.frames, self.feat_dim, dtype=torch.float32)
            edges = []
            for i in range(self.frames - 1):
                edges.append([i, i + 1])
                edges.append([i + 1, i])
            edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous() if len(edges) else torch.empty((2, 0), dtype=torch.long)
            y_w = torch.tensor(0.0)
            y_c = torch.tensor(0.0)
            return Data(x=x, edge_index=edge_index, y_wheeze=y_w, y_crackle=y_c)

    val_ds = SmallDataset(n=6, frames=8, feat_dim=768)
    val_loader = DataLoader(val_ds, batch_size=2, shuffle=False)
    print('Created synthetic val_loader with', len(val_ds), 'graphs')
else:
    print('val_loader already present')

if 'TRAIN_DEVICE' not in globals():
    import torch
    TRAIN_DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print('Set TRAIN_DEVICE ->', TRAIN_DEVICE)


In [ ]:
# Ensure project root is on sys.path so local packages (web_deploy, model_comparisons) import correctly
import sys, pathlib
if 'proj_root' in globals():
    repo_root = pathlib.Path(proj_root)
else:
    repo_root = pathlib.Path().resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
print('Ensured repo root on sys.path ->', str(repo_root))


In [ ]:
# Evaluation visualizations: ROC/PR, calibration, distributions, PCA, and attention bars
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import torch

# sklearn imports (optional fallback handling)
try:
    from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score
    from sklearn.calibration import calibration_curve
    from sklearn.decomposition import PCA
    from sklearn.metrics import confusion_matrix
    from sklearn.manifold import TSNE
except Exception:
    roc_curve = auc = precision_recall_curve = average_precision_score = None
    calibration_curve = None
    PCA = None
    TSNE = None

# pick a model available in the notebook/kernel
if 'dm' in globals():
    model_for_eval = dm
elif 'trained_model_full' in globals():
    model_for_eval = trained_model_full
elif 'trained_model' in globals():
    model_for_eval = trained_model
elif 'model' in globals():
    model_for_eval = model
else:
    # attempt to load a checkpoint via deploy utils
    try:
        import web_deploy.deploy_utils as deploy_utils
        ckpt = 'best_improved_30epochs_no_earlystop.pt'
        if not os.path.exists(ckpt):
            raise FileNotFoundError('No checkpoint found: ' + ckpt)
        model_for_eval = deploy_utils.load_deployment_model(ckpt, device='cpu')
    except Exception as e:
        raise RuntimeError('No model available for visualization: ' + repr(e))

# device
device = torch.device('cpu')

def collect_preds_and_embeddings(model, loader, device='cpu'):
    """Compute per-graph probabilities, labels, pooled embeddings and attention weights.

    Returns:
        w_probs, c_probs, w_trues, c_trues, embeddings (N x D), attn_list (list length N of 1D arrays), node_counts (list)
    """
    model.to(device)
    model.eval()
    all_w_probs, all_c_probs, all_w_trues, all_c_trues = [], [], [], []
    all_embeddings = []
    all_attn = []
    node_counts = []

    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)

            # replicate the forward up to the attention pooling to extract node-level features
            x = model.input_proj(batch.x)
            residuals = []
            for i, (conv, norm) in enumerate(zip(model.gat_layers, model.norms)):
                x_new = conv(x, batch.edge_index)
                x_new = torch.nn.functional.elu(x_new)
                x_new = norm(x_new)
                if i > 0 and i % 2 == 0:
                    x_new = x_new + residuals[-1]
                x = torch.nn.functional.dropout(x_new, p=model.dropout, training=model.training)
                residuals.append(x)

            attn_scores = model.attention_pool(x).squeeze(-1)

            # per-graph pooling and collect attention weights
            x_graph = []
            attn_graphs = []
            for b in torch.unique(batch.batch):
                mask = batch.batch == b
                scores = attn_scores[mask]
                weights = torch.softmax(scores, dim=0).cpu().numpy()
                pooled = (x[mask] * torch.softmax(scores, dim=0).unsqueeze(-1)).sum(dim=0)
                x_graph.append(pooled.cpu().numpy())
                attn_graphs.append(weights)
                node_counts.append(int(mask.sum().cpu().numpy()))

            x_graph = np.stack(x_graph, axis=0) if len(x_graph) > 0 else np.zeros((0, 1))

            # model predictions
            w_logits, c_logits = model(batch)
            w_prob = torch.sigmoid(w_logits).cpu().numpy()
            c_prob = torch.sigmoid(c_logits).cpu().numpy()

            if hasattr(batch, 'y_wheeze'):
                w_trues = batch.y_wheeze.cpu().numpy()
            else:
                w_trues = np.zeros_like(w_prob)

            if hasattr(batch, 'y_crackle'):
                c_trues = batch.y_crackle.cpu().numpy()
            else:
                c_trues = np.zeros_like(c_prob)

            all_w_probs.extend(w_prob.tolist())
            all_c_probs.extend(c_prob.tolist())
            all_w_trues.extend(w_trues.tolist())
            all_c_trues.extend(c_trues.tolist())
            all_embeddings.append(x_graph)
            all_attn.extend(attn_graphs)

    embeddings = np.concatenate(all_embeddings, axis=0) if len(all_embeddings) > 0 else np.zeros((0, 1))
    return (np.array(all_w_probs), np.array(all_c_probs), np.array(all_w_trues), np.array(all_c_trues), embeddings, all_attn, node_counts)

# collect
w_probs, c_probs, w_trues, c_trues, embeddings, attn_list, node_counts = collect_preds_and_embeddings(model_for_eval, val_loader, device=device)
print('Collected', len(w_probs), 'graphs; embeddings shape', embeddings.shape)

# Plot ROC / PR
plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
if roc_curve is not None:
    try:
        fpr, tpr, _ = roc_curve(w_trues, w_probs)
        auc_w = auc(fpr, tpr)
        plt.plot(fpr, tpr, label=f'Wheeze AUC={auc_w:.3f}')
    except Exception:
        plt.text(0.5,0.5,'Wheeze ROC failed')
    try:
        fpr2, tpr2, _ = roc_curve(c_trues, c_probs)
        auc_c = auc(fpr2, tpr2)
        plt.plot(fpr2, tpr2, label=f'Crackle AUC={auc_c:.3f}')
    except Exception:
        pass
    plt.plot([0,1],[0,1],'--', color='gray')
    plt.legend()
else:
    plt.text(0.5,0.5,'sklearn.metrics missing')
plt.title('ROC curves')

plt.subplot(1,2,2)
if precision_recall_curve is not None:
    try:
        prec, rec, _ = precision_recall_curve(w_trues, w_probs)
        ap = average_precision_score(w_trues, w_probs)
        plt.plot(rec, prec, label=f'Wheeze AP={ap:.3f}')
    except Exception:
        plt.text(0.5,0.5,'Wheeze PR failed')
    try:
        prec2, rec2, _ = precision_recall_curve(c_trues, c_probs)
        ap2 = average_precision_score(c_trues, c_probs)
        plt.plot(rec2, prec2, label=f'Crackle AP={ap2:.3f}')
    except Exception:
        pass
    plt.legend()
else:
    plt.text(0.5,0.5,'sklearn.metrics missing')
plt.title('Precision-Recall')
plt.tight_layout()
plt.show()

# Calibration and distributions
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
if calibration_curve is not None:
    try:
        prob_true, prob_pred = calibration_curve(w_trues, w_probs, n_bins=10)
        plt.plot(prob_pred, prob_true, marker='o', label='Wheeze')
    except Exception:
        plt.text(0.5,0.5,'Wheeze calibration failed')
    try:
        prob_true2, prob_pred2 = calibration_curve(c_trues, c_probs, n_bins=10)
        plt.plot(prob_pred2, prob_true2, marker='o', label='Crackle')
    except Exception:
        pass
    plt.plot([0,1],[0,1],'k--')
    plt.xlabel('Predicted probability')
    plt.ylabel('Fraction of positives')
    plt.legend()
else:
    plt.text(0.5,0.5,'sklearn.calibration missing')

plt.subplot(1,2,2)
sns.histplot(w_probs, color='C0', label='Wheeze', stat='density', bins=20)
sns.histplot(c_probs, color='C1', label='Crackle', stat='density', bins=20, alpha=0.6)
plt.legend()
plt.title('Predicted probability distributions')
plt.tight_layout()
plt.show()

# PCA on graph embeddings (wheeze coloring)
if embeddings is not None and embeddings.shape[0] > 1 and embeddings.shape[1] > 1 and PCA is not None:
    try:
        pca = PCA(n_components=2)
        z = pca.fit_transform(embeddings)
        plt.figure(figsize=(6,5))
        sc = plt.scatter(z[:,0], z[:,1], c=w_trues, cmap='coolwarm', s=50, edgecolor='k', alpha=0.85)
        plt.colorbar(sc, label='Wheeze label (0/1)')
        plt.title('PCA of graph embeddings colored by wheeze label')
        plt.show()
    except Exception as e:
        print('PCA failed:', e)
else:
    print('Embeddings too small or sklearn missing for PCA')

# Attention bars for first few graphs
n_display = min(6, len(attn_list))
for i in range(n_display):
    weights = attn_list[i]
    plt.figure(figsize=(7,2))
    plt.bar(np.arange(len(weights)), weights, color='C0')
    plt.xlabel('Node/frame index')
    plt.ylabel('Attention weight')
    plt.title(f'Graph {i}: nodes={node_counts[i]} pred_w={w_probs[i]:.3f} true_w={int(w_trues[i])} pred_c={c_probs[i]:.3f} true_c={int(c_trues[i])}')
    plt.tight_layout()
    plt.show()

print('Visualization generation complete.')


In [ ]:
# Saliency vs attention, confusion matrices, and t-SNE overlays
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from torch_geometric.data import Data

# pick first graph from a validation batch
batch_sample = next(iter(val_loader))
mask = batch_sample.batch == 0
if int(mask.sum()) == 0:
    print('Graph 0 empty; cannot compute saliency.')
else:
    node_idx = mask.nonzero(as_tuple=False).squeeze(-1).cpu().numpy()
    edge_index_np = batch_sample.edge_index.cpu().numpy()
    cols_mask = np.isin(edge_index_np[0], node_idx) & np.isin(edge_index_np[1], node_idx)
    sub_edges = edge_index_np[:, cols_mask].copy()
    idx_map = {old: int(i) for i, old in enumerate(node_idx)}
    reindexed = np.vectorize(idx_map.get)(sub_edges)

    x = batch_sample.x[mask].clone().detach()
    x = x.requires_grad_(True)
    data_single = Data(x=x, edge_index=torch.tensor(reindexed, dtype=torch.long))

    model_for_eval.to('cpu')
    model_for_eval.eval()

    # forward and compute gradients w.r.t node features for wheeze output
    w_logits, c_logits = model_for_eval(data_single)
    w_scalar = w_logits.squeeze() if hasattr(w_logits, 'squeeze') else w_logits
    if getattr(w_scalar, 'numel', lambda: 1)() > 1:
        w_scalar = w_scalar[0]

    grads = torch.autograd.grad(w_scalar, data_single.x, retain_graph=True)[0]
    saliency = grads.norm(p=2, dim=1).cpu().numpy()

    # attention weights for the same nodes
    with torch.no_grad():
        x_proj = model_for_eval.input_proj(data_single.x)
        attn_scores = model_for_eval.attention_pool(x_proj).squeeze(-1)
        attn_weights = torch.softmax(attn_scores, dim=0).cpu().numpy()

    # normalize for comparison
    sal_norm = saliency / (saliency.sum() + 1e-12)
    attn_norm = attn_weights / (attn_weights.sum() + 1e-12)
    corr = np.corrcoef(sal_norm, attn_norm)[0, 1]
    print(f'Pearson corr (saliency vs attention) = {corr:.3f}')

    # plot
    plt.figure(figsize=(8,3))
    ax = plt.gca()
    ax.bar(np.arange(len(attn_norm)), attn_norm, label='Attention', alpha=0.6)
    ax2 = ax.twinx()
    ax2.plot(np.arange(len(sal_norm)), sal_norm, color='C3', marker='o', label='Saliency (L2 norm)')
    ax.set_xlabel('Node/frame index')
    ax.set_ylabel('Attention (normed)')
    ax2.set_ylabel('Saliency (normed)')
    ax.set_title(f'Graph 0: attn vs saliency corr={corr:.3f}')
    ax.legend(loc='upper left')
    ax2.legend(loc='upper right')
    plt.tight_layout()
    plt.show()

# Confusion matrices for wheeze and crackle
thr = 0.5
w_pred = (w_probs >= thr).astype(int)
c_pred = (c_probs >= thr).astype(int)

try:
    from sklearn.metrics import confusion_matrix
    cm_w = confusion_matrix(w_trues, w_pred, labels=[0, 1])
    cm_c = confusion_matrix(c_trues, c_pred, labels=[0, 1])

    plt.figure(figsize=(10,4))
    plt.subplot(1,2,1)
    sns.heatmap(cm_w, annot=True, fmt='d', cmap='Blues')
    plt.title('Wheeze Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('True')

    plt.subplot(1,2,2)
    sns.heatmap(cm_c, annot=True, fmt='d', cmap='Blues')
    plt.title('Crackle Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.tight_layout()
    plt.show()
except Exception as e:
    print('Confusion matrix plotting failed:', e)

# t-SNE of embeddings colored by predicted wheeze
if embeddings is not None and embeddings.shape[0] > 1:
    try:
        from sklearn.manifold import TSNE
        z = TSNE(n_components=2, random_state=42).fit_transform(embeddings)
        plt.figure(figsize=(6,5))
        plt.scatter(z[:, 0], z[:, 1], c=(w_pred), cmap='coolwarm', s=80, edgecolor='k')
        plt.title('t-SNE of graph embeddings colored by predicted wheeze')
        plt.colorbar(label='Predicted wheeze (0/1)')
        plt.show()
    except Exception as e:
        print('t-SNE failed:', e)

# Attention centroid vs predicted probability
centroids = []
for att in attn_list:
    idx = np.arange(len(att))
    centroids.append((att * idx).sum() / (att.sum() + 1e-12))
centroids = np.array(centroids)
if len(centroids) == len(w_probs):
    corr_centroid = np.corrcoef(centroids, w_probs)[0, 1]
    print('Corr between attention centroid and wheeze prob:', corr_centroid)

    plt.figure(figsize=(6,3))
    plt.scatter(centroids, w_probs, c=w_trues, cmap='coolwarm', s=80, edgecolor='k')
    plt.xlabel('Attention centroid (index)')
    plt.ylabel('Wheeze predicted prob')
    plt.title('Attention centroid vs wheeze probability')
    plt.colorbar(label='True wheeze')
    plt.show()
else:
    print('Centroid analysis skipped: sizes mismatch')


In [ ]:
# End-to-end evaluation + visualization (full validation) and single-file audio test
# - Runs evaluation and visualizations on `val_loader` (must exist)
# - Tests the GNN on `model_comparisons/audio_test_files/sounds-of-asthma-wheezing-lung-sounds.mp3`
# - Saves visualizations to `output files/visualizations/`

import os
import sys
import pathlib
import numpy as np
import torch
import matplotlib.pyplot as plt
import librosa

out_dir = os.path.join('output files', 'visualizations')
os.makedirs(out_dir, exist_ok=True)

# ensure repo root on path
repo_root = pathlib.Path(globals().get('proj_root', pathlib.Path().resolve()))
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# pick or load model for evaluation
model_for_eval = None
if 'trained_model_full' in globals():
    model_for_eval = trained_model_full
elif 'trained_model' in globals():
    model_for_eval = trained_model
elif 'dm' in globals():
    model_for_eval = dm
else:
    try:
        from web_deploy.deploy_utils import load_deployment_model
        # prefer the full trained checkpoint name used earlier
        ckpt_candidates = [
            'best_improved_30epochs_no_earlystop.pt',
            'best_improved_30epochs.pt',
            'best_improved_30epochs_no_earlystop.pt',
        ]
        ckpt = next((c for c in ckpt_candidates if os.path.exists(c)), None)
        if ckpt is None:
            raise FileNotFoundError('No checkpoint found among candidates: ' + str(ckpt_candidates))
        model_for_eval = load_deployment_model(ckpt, device='cpu')
    except Exception as e:
        raise RuntimeError('No model available for evaluation: ' + repr(e))

print('Model for eval device:', next(model_for_eval.parameters()).device, 'params:', sum(p.numel() for p in model_for_eval.parameters()))

# require val_loader
if 'val_loader' not in globals():
    raise RuntimeError('`val_loader` not found in notebook. Please create a validation DataLoader and rerun this cell.')

# (re)define a lightweight collector (safe to re-run)
def collect_preds_and_embeddings(model, loader, device='cpu'):
    model.to(device)
    model.eval()
    all_w_probs, all_c_probs, all_w_trues, all_c_trues = [], [], [], []
    all_embeddings = []
    all_attn = []
    node_counts = []

    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            # forward until node features processed
            x = model.input_proj(batch.x)
            residuals = []
            for i, (conv, norm) in enumerate(zip(model.gat_layers, model.norms)):
                x_new = conv(x, batch.edge_index)
                x_new = torch.nn.functional.elu(x_new)
                x_new = norm(x_new)
                if i > 0 and i % 2 == 0:
                    x_new = x_new + residuals[-1]
                x = torch.nn.functional.dropout(x_new, p=model.dropout, training=model.training)
                residuals.append(x)

            attn_scores = model.attention_pool(x).squeeze(-1)

            x_graph = []
            attn_graphs = []
            for b in torch.unique(batch.batch):
                mask = batch.batch == b
                scores = attn_scores[mask]
                weights = torch.softmax(scores, dim=0).cpu().numpy()
                pooled = (x[mask] * torch.softmax(scores, dim=0).unsqueeze(-1)).sum(dim=0)
                x_graph.append(pooled.cpu().numpy())
                attn_graphs.append(weights)
                node_counts.append(int(mask.sum().cpu().numpy()))

            x_graph = np.stack(x_graph, axis=0) if len(x_graph) > 0 else np.zeros((0, 1))

            w_logits, c_logits = model(batch)
            w_prob = torch.sigmoid(w_logits).cpu().numpy()
            c_prob = torch.sigmoid(c_logits).cpu().numpy()

            if hasattr(batch, 'y_wheeze'):
                w_trues = batch.y_wheeze.cpu().numpy()
            else:
                w_trues = np.zeros_like(w_prob)

            if hasattr(batch, 'y_crackle'):
                c_trues = batch.y_crackle.cpu().numpy()
            else:
                c_trues = np.zeros_like(c_prob)

            all_w_probs.extend(w_prob.tolist())
            all_c_probs.extend(c_prob.tolist())
            all_w_trues.extend(w_trues.tolist())
            all_c_trues.extend(c_trues.tolist())
            all_embeddings.append(x_graph)
            all_attn.extend(attn_graphs)

    embeddings = np.concatenate(all_embeddings, axis=0) if len(all_embeddings) > 0 else np.zeros((0, 1))
    return (np.array(all_w_probs), np.array(all_c_probs), np.array(all_w_trues), np.array(all_c_trues), embeddings, all_attn, node_counts)

# collect preds
w_probs, c_probs, w_trues, c_trues, embeddings, attn_list, node_counts = collect_preds_and_embeddings(model_for_eval, val_loader, device='cpu')
print('Collected', len(w_probs), 'graphs; embeddings shape', embeddings.shape)

# run existing evaluate helper if available
try:
    from model_comparisons.train_improved import evaluate_with_postprocessing
    metrics = evaluate_with_postprocessing(model_for_eval, val_loader, device='cpu', threshold=0.5)
    print('Evaluation metrics (threshold=0.5):', metrics)
except Exception:
    print('evaluate_with_postprocessing not available or failed; continuing with collected preds')

# Save a simple ROC/PR and attention summary (reuse matplotlib/plt)
import matplotlib.pyplot as plt
from pathlib import Path
save_root = Path(out_dir)

# ROC & PR (sklearn optional)
try:
    from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score
    plt.figure(figsize=(10,4))
    plt.subplot(1,2,1)
    fpr, tpr, _ = roc_curve(w_trues, w_probs)
    plt.plot(fpr, tpr, label=f'Wheeze AUC={auc(fpr,tpr):.3f}')
    fpr2, tpr2, _ = roc_curve(c_trues, c_probs)
    plt.plot(fpr2, tpr2, label=f'Crackle AUC={auc(fpr2,tpr2):.3f}')
    plt.plot([0,1],[0,1],'--',color='gray')
    plt.legend()
    plt.title('ROC')

    plt.subplot(1,2,2)
    prec, rec, _ = precision_recall_curve(w_trues, w_probs)
    plt.plot(rec, prec, label=f'Wheeze AP={average_precision_score(w_trues,w_probs):.3f}')
    prec2, rec2, _ = precision_recall_curve(c_trues, c_probs)
    plt.plot(rec2, prec2, label=f'Crackle AP={average_precision_score(c_trues,c_probs):.3f}')
    plt.legend()
    plt.title('Precision-Recall')
    plt.tight_layout()
    plt.savefig(save_root / 'roc_pr.png')
    plt.show()
except Exception as e:
    print('ROC/PR plotting skipped:', e)

# Attention summary plot for first N graphs
N = min(6, len(attn_list))
for i in range(N):
    plt.figure(figsize=(6,2))
    plt.bar(np.arange(len(attn_list[i])), attn_list[i], color='C0')
    plt.title(f'Graph {i} attention (nodes={node_counts[i]}) pred_w={w_probs[i]:.3f} true_w={int(w_trues[i])}')
    plt.tight_layout()
    p = save_root / f'attention_graph_{i}.png'
    plt.savefig(p)
    plt.show()

print('Saved visualizations to', save_root)

# --- Single audio file test ---
mp3_path = os.path.join('model_comparisons', 'audio_test_files', 'sounds-of-asthma-wheezing-lung-sounds.mp3')
if not os.path.exists(mp3_path):
    print('Test audio not found at', mp3_path)
else:
    print('Running single-file test on', mp3_path)
    wav, sr = librosa.load(mp3_path, sr=16000, mono=True)
    frame_len = int(0.5 * sr)
    frames = []
    for i in range(0, len(wav), frame_len):
        f = wav[i:i+frame_len]
        if len(f) < frame_len:
            f = np.pad(f, (0, frame_len - len(f)))
        frames.append(f.astype(np.float32))

    # compute frame embeddings using available wrapper if present
    embeddings_list = []
    if 'wav2vec_frame_embedding_safe' in globals():
        for fr in frames:
            emb = wav2vec_frame_embedding_safe(fr, sr)
            embeddings_list.append(emb)
    else:
        # fallback: use HuggingFace wav2vec2 (may download weights)
        print('Using HF wav2vec2 fallback (may download model weights)')
        from transformers import Wav2Vec2Processor, Wav2Vec2Model
        proc = Wav2Vec2Processor.from_pretrained('facebook/wav2vec2-base-960h')
        w2v = Wav2Vec2Model.from_pretrained('facebook/wav2vec2-base-960h')
        w2v.eval()
        for fr in frames:
            inputs = proc(fr, sampling_rate=sr, return_tensors='pt', padding=True)
            with torch.no_grad():
                out = w2v(inputs.input_values)
                emb = out.last_hidden_state.mean(dim=1).squeeze(0).cpu().numpy()
                embeddings_list.append(emb.astype(np.float32))

    if len(embeddings_list) == 0:
        print('No embeddings computed; aborting audio test')
    else:
        X = np.stack(embeddings_list, axis=0)
        # build bidirectional edge_index between consecutive frames
        edges = []
        for i in range(X.shape[0] - 1):
            edges.append([i, i+1]); edges.append([i+1, i])
        edge_index = np.array(edges, dtype=np.int64).T if len(edges) else np.zeros((2,0), dtype=np.int64)

        try:
            from web_deploy.deploy_utils import predict_from_arrays
            from web_deploy.deploy_utils import load_deployment_model
            # ensure model loaded
            if model_for_eval is None:
                model_for_eval = load_deployment_model(ckpt, device='cpu')

            w_p, c_p = predict_from_arrays(model_for_eval, X, edge_index, device='cpu')
            print('Single-file predicted wheeze_prob:', w_p, 'crackle_prob:', c_p)

            # compute attention weights for plotting
            with torch.no_grad():
                x_t = torch.tensor(X, dtype=torch.float32)
                x_proj = model_for_eval.input_proj(x_t)
                attn_scores = model_for_eval.attention_pool(x_proj).squeeze(-1)
                attn_weights = torch.softmax(attn_scores, dim=0).cpu().numpy()

            # plot spectrogram + attention
            plt.figure(figsize=(10,4))
            plt.subplot(2,1,1)
            plt.specgram(wav, NFFT=512, Fs=sr, noverlap=256, cmap='magma')
            plt.title('Spectrogram with attention overlay')

            plt.subplot(2,1,2)
            idxs = np.arange(len(attn_weights))
            plt.bar(idxs, attn_weights, color='C1')
            plt.xlabel('Frame index (~0.5s per frame)')
            plt.ylabel('Attention weight')
            plt.tight_layout()
            plt.savefig(save_root / 'audio_test_attention.png')
            plt.show()

        except Exception as e:
            print('Audio test failed:', e)

print('End-to-end evaluation + single-file test complete.')


In [ ]:
# Single-file audio test (absolute path) — uses the provided MP3 path
import os
import numpy as np
import torch
import librosa
import matplotlib.pyplot as plt
from pathlib import Path

mp3_path = r"E:\studies\FINAL YR PROJECT\model\model_comparisons\audio_test_files\sounds-of-asthma-wheezing-lung-sounds.mp3"
print('Test audio path:', mp3_path)

if not os.path.exists(mp3_path):
    print('File not found at', mp3_path)
else:
    wav, sr = librosa.load(mp3_path, sr=16000, mono=True)
    frame_len = int(0.5 * sr)
    frames = []
    for i in range(0, len(wav), frame_len):
        f = wav[i:i+frame_len]
        if len(f) < frame_len:
            f = np.pad(f, (0, frame_len - len(f)))
        frames.append(f.astype(np.float32))

    # compute frame embeddings
    embeddings_list = []
    if 'wav2vec_frame_embedding_safe' in globals():
        for fr in frames:
            emb = wav2vec_frame_embedding_safe(fr, sr)
            embeddings_list.append(emb)
    else:
        try:
            from transformers import Wav2Vec2Processor, Wav2Vec2Model
            proc = Wav2Vec2Processor.from_pretrained('facebook/wav2vec2-base-960h')
            w2v = Wav2Vec2Model.from_pretrained('facebook/wav2vec2-base-960h')
            w2v.eval()
            for fr in frames:
                inputs = proc(fr, sampling_rate=sr, return_tensors='pt', padding=True)
                with torch.no_grad():
                    out = w2v(inputs.input_values)
                    emb = out.last_hidden_state.mean(dim=1).squeeze(0).cpu().numpy()
                    embeddings_list.append(emb.astype(np.float32))
        except Exception as e:
            print('Wav2Vec fallback failed:', e)

    if len(embeddings_list) == 0:
        print('No embeddings computed; aborting audio test')
    else:
        X = np.stack(embeddings_list, axis=0)
        edges = []
        for i in range(X.shape[0] - 1):
            edges.append([i, i+1]); edges.append([i+1, i])
        edge_index = np.array(edges, dtype=np.int64)

        # load model if needed
        model_for_eval = globals().get('model_for_eval', None)
        if model_for_eval is None:
            try:
                from web_deploy.deploy_utils import load_deployment_model
                ckpt_candidates = [
                    'best_improved_30epochs_no_earlystop.pt',
                    'best_improved_30epochs.pt',
                ]
                ckpt = next((c for c in ckpt_candidates if os.path.exists(c)), None)
                if ckpt is None:
                    raise FileNotFoundError('No checkpoint found among candidates')
                model_for_eval = load_deployment_model(ckpt, device='cpu')
            except Exception as e:
                raise RuntimeError('Could not load model_for_eval: ' + repr(e))

        # run prediction
        from web_deploy.deploy_utils import predict_from_arrays
        w_p, c_p = predict_from_arrays(model_for_eval, X, edge_index.T, device='cpu')
        print('Predicted wheeze_prob:', w_p, 'crackle_prob:', c_p)

        # attention overlay
        with torch.no_grad():
            x_t = torch.tensor(X, dtype=torch.float32)
            x_proj = model_for_eval.input_proj(x_t)
            attn_scores = model_for_eval.attention_pool(x_proj).squeeze(-1)
            attn_weights = torch.softmax(attn_scores, dim=0).cpu().numpy()

        # plot spectrogram + attention
        plt.figure(figsize=(10,4))
        plt.subplot(2,1,1)
        plt.specgram(wav, NFFT=512, Fs=sr, noverlap=256, cmap='magma')
        plt.title('Spectrogram of test audio')

        plt.subplot(2,1,2)
        idxs = np.arange(len(attn_weights))
        plt.bar(idxs, attn_weights, color='C1')
        plt.xlabel('Frame index (~0.5s per frame)')
        plt.ylabel('Attention weight')
        plt.tight_layout()

        out_dir = Path('output files') / 'visualizations'
        out_dir.mkdir(parents=True, exist_ok=True)
        save_path = out_dir / 'audio_test_attention_absolute_path.png'
        plt.savefig(save_path)
        plt.show()
        print('Saved attention plot to', str(save_path))


In [ ]:
# Highlight top-K attention frames on the spectrogram and save result
# Quick inspection: highlights the top-K attention frames from the last audio test.
import os
import numpy as np
import torch
import librosa
import matplotlib.pyplot as plt
from pathlib import Path

mp3_path = globals().get('mp3_path', r"E:\studies\FINAL YR PROJECT\model\model_comparisons\audio_test_files\sounds-of-asthma-wheezing-lung-sounds.mp3")
if not os.path.exists(mp3_path):
    print('MP3 not found at', mp3_path)
else:
    # load waveform (if not already in kernel)
    if 'wav' not in globals() or 'sr' not in globals() or globals().get('mp3_path', None) != mp3_path:
        wav, sr = librosa.load(mp3_path, sr=16000, mono=True)
    else:
        wav = globals()['wav']; sr = globals()['sr']

    frame_len = int(0.5 * sr)

    # Use existing attention weights if present, otherwise compute them
    if 'attn_weights' in globals():
        attn = np.array(globals()['attn_weights'])
    else:
        # compute embeddings and attention via model_for_eval
        print('Computing frame embeddings and attention (may download wav2vec)...')
        frames = []
        for i in range(0, len(wav), frame_len):
            f = wav[i:i+frame_len]
            if len(f) < frame_len:
                f = np.pad(f, (0, frame_len - len(f)))
            frames.append(f.astype(np.float32))

        embeddings_list = []
        if 'wav2vec_frame_embedding_safe' in globals():
            for fr in frames:
                embeddings_list.append(wav2vec_frame_embedding_safe(fr, sr))
        else:
            from transformers import Wav2Vec2Processor, Wav2Vec2Model
            proc = Wav2Vec2Processor.from_pretrained('facebook/wav2vec2-base-960h')
            w2v = Wav2Vec2Model.from_pretrained('facebook/wav2vec2-base-960h')
            w2v.eval()
            for fr in frames:
                inputs = proc(fr, sampling_rate=sr, return_tensors='pt', padding=True)
                with torch.no_grad():
                    out = w2v(inputs.input_values)
                    emb = out.last_hidden_state.mean(dim=1).squeeze(0).cpu().numpy()
                    embeddings_list.append(emb.astype(np.float32))

        X = np.stack(embeddings_list, axis=0)
        # compute attention
        x_t = torch.tensor(X, dtype=torch.float32)
        x_proj = model_for_eval.input_proj(x_t)
        attn_scores = model_for_eval.attention_pool(x_proj).squeeze(-1)
        attn = torch.softmax(attn_scores, dim=0).cpu().numpy()

    # choose top-K
    K = 5
    topk_idx = np.argsort(attn)[-K:][::-1]

    # compute spectrogram (db) for plotting
    try:
        import librosa.display
        S = np.abs(librosa.stft(wav, n_fft=1024, hop_length=256))
        S_db = librosa.amplitude_to_db(S + 1e-6)
        times = np.linspace(0, len(wav) / sr, S_db.shape[1])
    except Exception:
        S_db = None

    plt.figure(figsize=(12,5))
    if S_db is not None:
        librosa.display.specshow(S_db, sr=sr, hop_length=256, x_axis='time', y_axis='hz', cmap='magma')
        plt.colorbar(format='%+2.0f dB')
    else:
        plt.specgram(wav, NFFT=512, Fs=sr, noverlap=256, cmap='magma')

    # overlay top-K frame spans
    for idx in topk_idx:
        start = idx * frame_len / float(sr)
        end = (idx + 1) * frame_len / float(sr)
        plt.axvspan(start, end, color='yellow', alpha=0.4)
        plt.text(start, 0.95, f'#{idx}', color='k', fontsize=10, weight='bold', transform=plt.gca().get_xaxis_transform())

    plt.title(f'Top-{K} attention frames overlay')
    out_dir = Path('output files') / 'visualizations'
    out_dir.mkdir(parents=True, exist_ok=True)
    save_path = out_dir / 'audio_test_attention_topk.png'
    plt.tight_layout()
    plt.savefig(save_path)
    plt.show()
    print('Saved top-K overlay to', save_path)


In [ ]:
# Integrated Gradients and Gradient×Input across positive examples
# This is compute-heavy. Default: max_samples=20, steps=25.
import time
import os
import numpy as np
import torch
import matplotlib.pyplot as plt
from pathlib import Path

max_samples = 20
steps = 25
target_task = 'wheeze'  # 'wheeze' or 'crackle'

if 'val_loader' not in globals():
    raise RuntimeError('val_loader not found; create a full validation loader before running integrated gradients.')

# helper to extract single-graph Data from batch and graph index
from torch_geometric.data import Data

def get_graph_from_batch(batch, graph_idx):
    mask = (batch.batch == graph_idx)
    x = batch.x[mask].detach().clone()
    # filter edges where both endpoints in mask
    ei = batch.edge_index.cpu().numpy()
    mask_ids = mask.nonzero(as_tuple=False).squeeze(-1).cpu().numpy()
    cols = np.isin(ei[0], mask_ids) & np.isin(ei[1], mask_ids)
    sub_edges = ei[:, cols]
    # re-index
    idx_map = {old: int(i) for i, old in enumerate(mask_ids)}
    if sub_edges.shape[1] > 0:
        reindexed = np.vectorize(idx_map.get)(sub_edges)
        edge_index = torch.tensor(reindexed, dtype=torch.long)
    else:
        edge_index = torch.empty((2,0), dtype=torch.long)
    return Data(x=x, edge_index=edge_index)

# Integrated gradients function
import torch.autograd as autograd

def integrated_gradients(model, data, target='wheeze', baseline=None, steps=25, device='cpu'):
    model.to(device)
    model.eval()
    x = data.x.to(device)
    edge_index = data.edge_index.to(device)

    if baseline is None:
        baseline = torch.zeros_like(x)

    grads_sum = torch.zeros_like(x)
    for alpha in np.linspace(0.0, 1.0, steps):
        x_scaled = (baseline + alpha * (x - baseline)).detach()
        x_scaled.requires_grad_()
        d = Data(x=x_scaled, edge_index=edge_index)
        w_logits, c_logits = model(d)
        out = w_logits if target == 'wheeze' else c_logits
        out_scalar = out.squeeze()
        if out_scalar.numel() > 1:
            out_scalar = out_scalar[0]
        grad = autograd.grad(out_scalar, x_scaled, retain_graph=False, create_graph=False)[0]
        grads_sum += grad

    avg_grads = grads_sum / float(steps)
    attributions = (x - baseline) * avg_grads
    # node-wise L2 norm of attribution
    node_attr = attributions.detach().cpu().norm(p=2, dim=1).numpy()
    return node_attr

# gradient x input (single-shot)
def grad_times_input(model, data, target='wheeze', device='cpu'):
    model.to(device)
    model.eval()
    x = data.x.detach().clone().to(device)
    x.requires_grad_()
    d = Data(x=x, edge_index=data.edge_index.to(device))
    w_logits, c_logits = model(d)
    out = w_logits if target == 'wheeze' else c_logits
    out_scalar = out.squeeze()
    if out_scalar.numel() > 1:
        out_scalar = out_scalar[0]
    grad = autograd.grad(out_scalar, x, retain_graph=False, create_graph=False)[0]
    gxi = (grad * x).detach().cpu().norm(p=2, dim=1).numpy()
    return gxi

# iterate val_loader and collect positive examples for target_task
pos_count = 0
ig_top_counts = {}
gxi_top_counts = {}
processed = 0
start_time = time.time()

for batch in val_loader:
    if processed >= max_samples:
        break
    # take each graph in batch
    n_graphs = int(batch.y_wheeze.shape[0]) if hasattr(batch, 'y_wheeze') else 0
    for gi in range(n_graphs):
        label = (batch.y_wheeze[gi].item() if target_task == 'wheeze' else batch.y_crackle[gi].item())
        if label < 0.5:
            continue
        # extract graph
        data = get_graph_from_batch(batch, gi)
        if data.x.shape[0] == 0:
            continue
        try:
            ig_attr = integrated_gradients(model_for_eval, data, target=target_task, baseline=None, steps=steps, device='cpu')
            gxi_attr = grad_times_input(model_for_eval, data, target=target_task, device='cpu')
        except Exception as e:
            print('Attribution failed for graph', gi, 'error:', e)
            continue

        # top-k nodes
        k = min(5, len(ig_attr))
        top_ig = np.argsort(ig_attr)[-k:][::-1]
        top_gxi = np.argsort(gxi_attr)[-k:][::-1]
        for t in top_ig:
            ig_top_counts[t] = ig_top_counts.get(t, 0) + 1
        for t in top_gxi:
            gxi_top_counts[t] = gxi_top_counts.get(t, 0) + 1

        pos_count += 1
        processed += 1
        if processed >= max_samples:
            break

elapsed = time.time() - start_time
print(f'Processed {pos_count} positive examples (max {max_samples}) in {elapsed:.1f}s')

# summarize top frames frequency (aggregated by node index)
if len(ig_top_counts) == 0:
    print('No positive examples found for integrated gradients. Aborting summary.')
else:
    # convert counts to sorted list
    items = sorted(ig_top_counts.items(), key=lambda x: x[1], reverse=True)
    print('Top IG nodes (index,count):', items[:10])
    items_g = sorted(gxi_top_counts.items(), key=lambda x: x[1], reverse=True)
    print('Top Grad×Input nodes (index,count):', items_g[:10])

    # plot frequency bar
    plt.figure(figsize=(8,4))
    idxs = [i for i,c in items]
    vals = [c for i,c in items]
    plt.bar(idxs[:20], vals[:20], color='C2')
    plt.title('Frequency of being top-K (Integrated Gradients)')
    plt.xlabel('Node/frame index')
    plt.ylabel('Count')
    save_root = Path('output files') / 'visualizations'
    save_root.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(save_root / 'integrated_gradients_top_freq.png')
    plt.show()

    # save CSV
    import csv
    csv_path = save_root / 'integrated_gradients_top_freq.csv'
    with open(csv_path, 'w', newline='') as fh:
        writer = csv.writer(fh)
        writer.writerow(['node_index','count'])
        for idx, cnt in items:
            writer.writerow([idx, cnt])
    print('Saved integrated gradients summary to', csv_path)


In [ ]:
# Analysis: validation label distribution, prediction metrics, and attention stats
# Saves a short text summary to `notebooks/output files/visualizations/analysis_summary.txt`
import numpy as np
from pathlib import Path
out_dir = Path('notebooks') / 'output files' / 'visualizations'
out_dir.mkdir(parents=True, exist_ok=True)
lines = []

def save_and_print(s):
    print(s)
    lines.append(str(s))

# 1) val_loader label counts
if 'val_loader' in globals():
    total = 0
    pos_w = 0
    pos_c = 0
    for batch in val_loader:
        if hasattr(batch, 'y_wheeze'):
            y = batch.y_wheeze
            pos_w += int(y.sum().item())
            total += y.shape[0]
        elif hasattr(batch, 'y'):
            y = batch.y
            if getattr(y, 'dim', lambda: 0)() == 2 and y.shape[1] >= 2:
                pos_w += int(y[:, 0].sum().item())
                pos_c += int(y[:, 1].sum().item())
                total += y.shape[0]
            else:
                total += y.shape[0]
        if hasattr(batch, 'y_crackle'):
            pos_c += int(batch.y_crackle.sum().item())
    save_and_print(f"Val loader: graphs={total}, pos_wheeze={pos_w}, pos_crackle={pos_c}")
else:
    save_and_print('val_loader not in kernel globals')

# 2) If prediction arrays exist, compute AUC/AP/confusion
try:
    from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix, classification_report
    sklearn_ok = True
except Exception:
    sklearn_ok = False
    save_and_print('scikit-learn not available in kernel; cannot compute AUC/AP')


def analyze_preds(y_true_arr, y_prob_arr, name):
    y_true = np.asarray(y_true_arr)
    y_prob = np.asarray(y_prob_arr)
    save_and_print(f"{name}: n={len(y_true)}, unique={np.unique(y_true)}")
    if len(np.unique(y_true)) == 1:
        save_and_print(f"{name}: single-class labels -> AUC/AP undefined. Positive count={int(y_true.sum())}")
        return
    if not sklearn_ok:
        return
    auc = roc_auc_score(y_true, y_prob)
    ap = average_precision_score(y_true, y_prob)
    pred_lab = (y_prob >= 0.5).astype(int)
    cm = confusion_matrix(y_true, pred_lab)
    report = classification_report(y_true, pred_lab, zero_division=0)
    save_and_print(f"{name}: AUC={auc:.4f}, AP={ap:.4f}")
    save_and_print("Confusion matrix:\n" + str(cm))
    save_and_print("Classification report:\n" + report)

# Wheeze
if 'w_trues' in globals() and 'w_probs' in globals():
    analyze_preds(w_trues, w_probs, 'Wheeze')
else:
    save_and_print('w_trues/w_probs not present in kernel')

# Crackle
if 'c_trues' in globals() and 'c_probs' in globals():
    analyze_preds(c_trues, c_probs, 'Crackle')
else:
    save_and_print('c_trues/c_probs not present in kernel')

# 3) Attention stats if available
if 'attn_weights' in globals():
    a = np.array(attn_weights)
    save_and_print(f"attn_weights shape={a.shape}")
    if a.ndim == 1:
        save_and_print(f"attn mean={a.mean():.4f} std={a.std():.4f}")
    else:
        save_and_print("attn per-node mean: " + str(np.round(a.mean(axis=0), 4)))
        save_and_print("attn per-node std: " + str(np.round(a.std(axis=0), 4)))
else:
    save_and_print('attn_weights not present in kernel')

# Save summary
fp = out_dir / 'analysis_summary.txt'
with open(fp, 'w') as fh:
    for l in lines:
        fh.write(l + '\n')
save_and_print(f"Saved analysis summary to {fp}")


In [ ]:
# Build stratified validation DataLoader from ICBHI metadata (lightweight subset)
# Safely handle missing `meta_df` and avoid heavy wav2vec work here.
import os
import random
import glob
import numpy as np
import torch
from pathlib import Path
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader

# Parameters (quick defaults)
val_max_files = 60
frame_len_s = 0.5
sr = 16000
frame_len = int(frame_len_s * sr)
target_pos_w = 20
target_pos_c = 20

print(f'Building stratified val set (max_files={val_max_files})')

# Helper to parse annotation file for wheeze/crackle flags
def _parse_ann_flags(ann_path):
    has_w = False
    has_c = False
    try:
        with open(ann_path, 'r') as fh:
            for ln in fh:
                parts = ln.strip().split()
                if len(parts) >= 4:
                    try:
                        w_flag = int(float(parts[2]))
                        c_flag = int(float(parts[3]))
                    except Exception:
                        continue
                    if w_flag > 0:
                        has_w = True
                    if c_flag > 0:
                        has_c = True
    except Exception:
        pass
    return has_w, has_c

entries = []

# Case A: use existing `meta_df` if present in kernel globals
if 'meta_df' in globals():
    try:
        for _, row in meta_df.iterrows():
            wav_path = row.get('wav_path') if hasattr(row, 'get') else None
            if not wav_path:
                try:
                    wav_path = row['wav']
                except Exception:
                    wav_path = None
            ann_path = row.get('ann_path') if hasattr(row, 'get') else None
            if not ann_path:
                try:
                    ann_path = row['ann']
                except Exception:
                    ann_path = None
            if not wav_path or not ann_path:
                continue
            if not (os.path.exists(wav_path) and os.path.exists(ann_path)):
                continue
            has_w, has_c = _parse_ann_flags(ann_path)
            entries.append({'wav': wav_path, 'ann': ann_path, 'w': has_w, 'c': has_c})
    except Exception as e:
        print('Error reading meta_df:', e)
else:
    print('meta_df not present in kernel globals; attempting repo scan for paired wav/.txt files')

# Case B: if still empty, try scanning a known dataset folder under project root
if len(entries) == 0:
    repo_root = Path(globals().get('proj_root', Path.cwd()))
    cand_dir = repo_root / 'ICBHI_final_database'
    if cand_dir.exists():
        wav_files = list(cand_dir.rglob('*.wav'))
        txt_files = list(cand_dir.rglob('*.txt'))
        wav_map = {p.stem: str(p) for p in wav_files}
        txt_map = {p.stem: str(p) for p in txt_files}
        paired = [(s, wav_map[s], txt_map[s]) for s in wav_map.keys() if s in txt_map]
        print('Found paired wav+txt under', cand_dir, 'count=', len(paired))
        for base, wav_path, ann_path in paired:
            has_w, has_c = _parse_ann_flags(ann_path)
            entries.append({'wav': wav_path, 'ann': ann_path, 'w': has_w, 'c': has_c})
    else:
        # fallback: scan repo for any paired wav/txt
        wav_files = list(repo_root.rglob('*.wav'))
        txt_files = list(repo_root.rglob('*.txt'))
        wav_map = {p.stem: str(p) for p in wav_files}
        txt_map = {p.stem: str(p) for p in txt_files}
        paired = [(s, wav_map[s], txt_map[s]) for s in wav_map.keys() if s in txt_map]
        print('Scanned repo root for pairs, found:', len(paired))
        for base, wav_path, ann_path in paired:
            has_w, has_c = _parse_ann_flags(ann_path)
            entries.append({'wav': wav_path, 'ann': ann_path, 'w': has_w, 'c': has_c})

# If still empty, do not raise — instruct user to create `meta_df` or use the synthetic fallback cell
if len(entries) == 0:
    print('No paired wav+annotation files found. Skipping stratified val build.')
    print('You can either create `meta_df` earlier in the notebook or run the synthetic val_loader cell (fast).')
else:
    # select files (stratified-ish)
    posw = [e for e in entries if e['w']]
    posc = [e for e in entries if e['c']]
    neg = [e for e in entries if not e['w'] and not e['c']]

    random.seed(42)
    selected = []

    num_pw = min(len(posw), target_pos_w)
    if num_pw > 0:
        selected.extend(random.sample(posw, num_pw))

    num_pc = min(len(posc), target_pos_c)
    if num_pc > 0:
        sampled_c = random.sample(posc, num_pc)
        for e in sampled_c:
            if e not in selected:
                selected.append(e)

    remaining = val_max_files - len(selected)
    if remaining > 0 and len(neg) > 0:
        neg_sample = random.sample(neg, min(len(neg), remaining))
        selected.extend(neg_sample)

    selected = selected[:val_max_files]
    print(f'Selected files: total={len(selected)} pos_w={sum(1 for e in selected if e["w"]) } pos_c={sum(1 for e in selected if e["c"]) } negatives={sum(1 for e in selected if not e["w"] and not e["c"]) }')

    # Build placeholder Data objects (use zero embeddings here to avoid heavy wav2vec compute)
    dataset = []
    feat_dim = 768
    placeholder_frames = 8
    for ent in selected:
        x = torch.zeros((placeholder_frames, feat_dim), dtype=torch.float32)
        edges = []
        for k in range(placeholder_frames - 1):
            edges.append([k, k + 1]); edges.append([k + 1, k])
        edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous() if len(edges) else torch.empty((2, 0), dtype=torch.long)
        data = Data(x=x, edge_index=edge_index, y_wheeze=torch.tensor(float(ent['w'])), y_crackle=torch.tensor(float(ent['c'])))
        dataset.append(data)

    val_loader = DataLoader(dataset, batch_size=8, shuffle=False)
    print('Built placeholder val_loader with', len(dataset), 'graphs; wheeze_pos=', int(sum(int(d.y_wheeze.item()) for d in dataset)), 'crackle_pos=', int(sum(int(d.y_crackle.item()) for d in dataset)))


In [ ]:
# QUICK: create a synthetic validation loader with positives for analysis (fast)
# This is a fallback because the full ICBHI audio files are not available in this workspace.
import random
import torch
import numpy as np
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader

n_val = 60
frames = 8
feat_dim = 768
pos_w_prob = 0.3
pos_c_prob = 0.25

dataset = []
for i in range(n_val):
    x = torch.randn(frames, feat_dim, dtype=torch.float32)
    edges = []
    for j in range(frames - 1):
        edges.append([j, j + 1]); edges.append([j + 1, j])
    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous() if len(edges) else torch.empty((2, 0), dtype=torch.long)
    y_w = 1.0 if random.random() < pos_w_prob else 0.0
    y_c = 1.0 if random.random() < pos_c_prob else 0.0
    dataset.append(Data(x=x, edge_index=edge_index, y_wheeze=torch.tensor(y_w), y_crackle=torch.tensor(y_c)))

val_loader = DataLoader(dataset, batch_size=8, shuffle=False)
print('Synthetic val_loader created ->', len(dataset), 'graphs; pos_w=', int(sum(int(d.y_wheeze.item()) for d in dataset)), 'pos_c=', int(sum(int(d.y_crackle.item()) for d in dataset)))


In [ ]:
# Example: Nurse-facing patient-state demo (streaming inference)
from web_deploy.deploy_utils import PatientStateManager, load_deployment_model, predict_with_metadata
import numpy as np
import time

# Instantiate a state manager (tweak thresholds/alpha as needed)
state_mgr = PatientStateManager(ema_alpha=0.12, low_delta=0.08, high_delta=0.20, min_samples_for_baseline=5)

# Load model (update path to your checkpoint)
model = load_deployment_model(r"..\web_deploy\../web_deploy/../web_deploy/../web_deploy/../web_deploy/../web_deploy/best_improved_30epochs_no_earlystop.pt", device="cpu")

# Dummy stream generator for testing: replace with real graph extractor in production
def dummy_graph(n_nodes=20, feat_dim=768):
    x = np.random.randn(n_nodes, feat_dim).astype(np.float32)
    # simple bidirectional chain edges
    edges = np.array([[i, i+1] for i in range(n_nodes-1)] + [[i+1, i] for i in range(n_nodes-1)], dtype=np.int64).T
    return x, edges

patient_id = "patient_001"
print("Streaming demo for", patient_id)
for step in range(8):
    x_arr, edge_idx = dummy_graph()
    out = predict_with_metadata(model, x_arr, edge_idx, device="cpu", patient_id=patient_id, state_manager=state_mgr, model_version="demo")
    print(f"Step {step+1}: STATE={out['patient_state']}  wheeze={out['wheeze_prob']:.3f}  crackle={out['crackle_prob']:.3f}")
    # concise reasoning for nurses
    rs = out.get('patient_state_reason')
    print(' Reason:', {k: {'state': v['state'], 'delta': round(v['delta'],3), 'trend': round(v['trend'],3)} for k,v in rs.items() if k in ['wheeze','crackle']})
    time.sleep(0.2)


# Test on multiple patient audio files (ICBHI samples + sample MP3)

In [ ]:
# Test on multiple patient audio files (ICBHI samples + sample MP3)
# Converts audio -> node features via Wav2Vec2, builds simple k-NN adjacency,
# runs inference and prints nurse-facing patient state.

from pathlib import Path
import sys
ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import librosa
import glob
from pathlib import Path
import numpy as np
import torch
from web_deploy.deploy_utils import PatientStateManager, load_deployment_model, predict_with_metadata

# helper: build k-NN edges via cosine similarity
def build_knn_edges(x_np, k=4):
    # x_np: [N, D]
    if x_np.shape[0] == 0:
        return np.zeros((2, 0), dtype=np.int64)
    x = x_np.astype(np.float32)
    norms = np.linalg.norm(x, axis=1, keepdims=True) + 1e-8
    xn = x / norms
    sim = xn @ xn.T
    N = x.shape[0]
    rows = []
    cols = []
    for i in range(N):
        # exclude self
        idx = np.argsort(-sim[i])
        picked = [j for j in idx if j != i][:k]
        for j in picked:
            rows.append(i)
            cols.append(j)
    if len(rows) == 0:
        return np.zeros((2, 0), dtype=np.int64)
    return np.vstack([np.array(rows, dtype=np.int64), np.array(cols, dtype=np.int64)])

# audio -> features using wav2vec (falls back to random if transformers missing)
def audio_to_graph(path, frame_seconds=0.5, sr=16000, k=4):
    try:
        from transformers import Wav2Vec2Processor, Wav2Vec2Model
        proc = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
        w2v = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base-960h")
        w2v.eval()
    except Exception as e:
        proc = None
        w2v = None

    y, _ = librosa.load(str(path), sr=sr, mono=True)
    frame_len = int(frame_seconds * sr)
    frames = []
    for i in range(0, len(y), frame_len):
        f = y[i:i+frame_len]
        if len(f) < frame_len:
            f = np.pad(f, (0, frame_len-len(f)), mode='constant')
        frames.append(f.astype(np.float32))
    if len(frames) == 0:
        return np.zeros((0,768), dtype=np.float32), np.zeros((2,0), dtype=np.int64)

    if proc is None or w2v is None:
        # fallback random features (deterministic by seed)
        rng = np.random.RandomState(0)
        x_np = rng.randn(len(frames), 768).astype(np.float32)
    else:
        feats = []
        with torch.no_grad():
            for f in frames:
                inputs = proc(f, sampling_rate=sr, return_tensors="pt", padding=True)
                out = w2v(**inputs)
                emb = out.last_hidden_state.mean(dim=1).squeeze(0).cpu().numpy()
                feats.append(emb.astype(np.float32))
        x_np = np.stack(feats, axis=0)

    edge_index = build_knn_edges(x_np, k=k)
    return x_np, edge_index

# pick files: 3 from ICBHI + the MP3
project_root = Path('..').resolve()
icbhi_dir = (project_root / "ICBHI_final_database")
mp3_path = project_root / "model_comparisons" / "audio_test_files" / "sounds-of-asthma-wheezing-lung-sounds.mp3"

wav_candidates = sorted(list(icbhi_dir.glob("*.wav")))[:3]
files = wav_candidates + ([mp3_path] if mp3_path.exists() else [])

print("Files to test:")
for f in files:
    print(" -", f)

# load model and state manager
ck = Path("../notebooks/best_improved_30epochs_no_earlystop.pt")
if not ck.exists():
    ck = Path("notebooks/best_improved_30epochs_no_earlystop.pt")
model = load_deployment_model(str(ck), device="cpu")
state_mgr = PatientStateManager(ema_alpha=0.12, low_delta=0.08, high_delta=0.20, min_samples_for_baseline=5)

# run through files
for p in files:
    pid = p.stem
    print('\n=== Patient:', pid, '===')
    x_np, edge_idx = audio_to_graph(p)
    if x_np.shape[0] == 0:
        print(' No frames extracted; skipping')
        continue
    out = predict_with_metadata(model, x_np, edge_idx, device='cpu', patient_id=pid, state_manager=state_mgr, model_version='best_improved_demo')
    print('Overall state:', out.get('patient_state'))
    rs = out.get('patient_state_reason')
    if rs:
        for axis in ('wheeze','crackle'):
            a = rs[axis]
            print(f" {axis}: state={a['state']} baseline={a['baseline']} value={a['value']:.3f} delta={a['delta']:.3f} trend={a['trend']:.3f}")


In [ ]:
# Sequential windowed streaming test to establish baselines and show state transitions
from pathlib import Path
import sys
ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import librosa
import numpy as np
import time
from web_deploy.deploy_utils import PatientStateManager, load_deployment_model, predict_with_metadata

# reuse audio_to_graph from previous cell by importing from notebook globals (if not, redefine minimal version)
try:
    audio_to_graph
except NameError:
    print('audio_to_graph not found; please run the earlier cell first.')

ck = Path("../notebooks/best_improved_30epochs_no_earlystop.pt")
if not ck.exists():
    ck = Path("notebooks/best_improved_30epochs_no_earlystop.pt")
model = load_deployment_model(str(ck), device="cpu")
state_mgr = PatientStateManager(ema_alpha=0.2, low_delta=0.08, high_delta=0.20, min_samples_for_baseline=4, force_established_after_s=10.0)

files = []
project_root = Path('..').resolve()
mp3 = project_root / "model_comparisons" / "audio_test_files" / "sounds-of-asthma-wheezing-lung-sounds.mp3"
if mp3.exists():
    files.append(mp3)
wav_candidates = sorted(list((project_root / 'ICBHI_final_database').glob('*.wav')))[:2]
files.extend(wav_candidates)

for p in files:
    print('\n--- Streaming windows for', p.stem, '---')
    duration = librosa.get_duration(filename=str(p))
    win = 5.0
    starts = np.arange(0.0, max(duration, 1e-6), win)
    for s in starts[:8]:
        # load window
        y, _ = librosa.load(str(p), sr=16000, mono=True, offset=float(s), duration=win)
        # write to temp array and process via wav2vec inside audio_to_graph-like logic
        # reuse audio_to_graph by creating a temporary path is expensive; instead use local embedding code
        try:
            from transformers import Wav2Vec2Processor, Wav2Vec2Model
            proc = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
            w2v = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base-960h")
            w2v.eval()
            frames = []
            frame_len = int(0.5 * 16000)
            for i in range(0, len(y), frame_len):
                f = y[i:i+frame_len]
                if len(f) < frame_len:
                    f = np.pad(f, (0, frame_len-len(f)), mode='constant')
                frames.append(f.astype(np.float32))
            feats = []
            import torch
            with torch.no_grad():
                for f in frames:
                    inputs = proc(f, sampling_rate=16000, return_tensors='pt', padding=True)
                    out = w2v(**inputs)
                    emb = out.last_hidden_state.mean(dim=1).squeeze(0).cpu().numpy()
                    feats.append(emb.astype(np.float32))
            x_np = np.stack(feats, axis=0)
            # build simple adjacency
            def build_knn_edges_local(x_np, k=4):
                if x_np.shape[0]==0:
                    return np.zeros((2,0), dtype=np.int64)
                xn = x_np / (np.linalg.norm(x_np,axis=1,keepdims=True)+1e-8)
                sim = xn @ xn.T
                rows=[]; cols=[]
                for i in range(x_np.shape[0]):
                    idx = np.argsort(-sim[i])
                    picked = [j for j in idx if j!=i][:k]
                    for j in picked:
                        rows.append(i); cols.append(j)
                if len(rows)==0:
                    return np.zeros((2,0),dtype=np.int64)
                return np.vstack([np.array(rows,dtype=np.int64), np.array(cols,dtype=np.int64)])
            edge_idx = build_knn_edges_local(x_np, k=4)
        except Exception:
            # fallback random features
            rng = np.random.RandomState(int(time.time())%123456)
            x_np = rng.randn(8,768).astype(np.float32)
            edge_idx = np.vstack([np.arange(8,dtype=np.int64), np.arange(8,dtype=np.int64)])

        out = predict_with_metadata(model, x_np, edge_idx, device='cpu', patient_id=p.stem, state_manager=state_mgr, model_version='demo')
        print(f"start={s:.1f}s state={out.get('patient_state')} wheeze={out['wheeze_prob']:.3f} crackle={out['crackle_prob']:.3f}")
        rs = out.get('patient_state_reason')
        if rs:
            print('  reason:', {k:{'state':v['state'],'delta':round(v['delta'],3)} for k,v in rs.items() if k in ['wheeze','crackle']})
        time.sleep(0.05)


In [ ]:
# Debug: direct state manager test (reload module to pick up edits)
import importlib
import web_deploy.deploy_utils as du
importlib.reload(du)
sm = du.PatientStateManager()
print(sm.update_and_get_state('dbg_patient', 0.3, 0.4))


# deployment ready run

In [ ]:
# Concrete end-to-end cell: raw audio -> wav2vec embeddings -> Data -> model outputs + breathing rate
# Includes cumulative, windowed reasoning to account for different audio lengths.
from pathlib import Path
import sys
import json
import time
import uuid
import datetime
import csv
import numpy as np
import torch
import librosa
from torch_geometric.data import Data

# Ensure repo root on sys.path for local imports
repo_root = Path('..').resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from web_deploy.deploy_utils import load_deployment_model, PatientStateManager

# Optional: attribution summaries (can be expensive)
compute_attributions = False
attribution_target = 'wheeze'  # 'wheeze' or 'crackle'
integrated_steps = 20
attr_topk = 3

# --- Select audio files (two ICBHI + the asthma MP3) ---
icbhi_dir = repo_root / 'ICBHI_final_database'
mp3_path = repo_root / 'model_comparisons' / 'audio_test_files' / 'sounds-of-asthma-wheezing-lung-sounds.mp3'

icbhi_files = sorted(icbhi_dir.glob('*.wav'))[:2] if icbhi_dir.exists() else []
files = icbhi_files + ([mp3_path] if mp3_path.exists() else [])

if len(files) == 0:
    raise FileNotFoundError('No audio files found to test. Check ICBHI_final_database and MP3 path.')

print('Testing files:')
for f in files:
    print(' -', f)

# --- Load model checkpoint ---
ckpt_candidates = [
    repo_root / 'notebooks' / 'best_improved_30epochs_no_earlystop.pt',
    repo_root / 'notebooks' / 'best_improved_30epochs.pt',
    repo_root / 'best_improved_30epochs_no_earlystop.pt',
]
ckpt = next((p for p in ckpt_candidates if p.exists()), None)
if ckpt is None:
    raise FileNotFoundError('No checkpoint found among: ' + ', '.join(str(p) for p in ckpt_candidates))

model = load_deployment_model(str(ckpt), device='cpu')
model.eval()

# --- Wav2Vec2 setup (reuse across files) ---
from transformers import Wav2Vec2Processor, Wav2Vec2Model

if '_W2V_PROC' not in globals() or '_W2V_MODEL' not in globals():
    _W2V_PROC = Wav2Vec2Processor.from_pretrained('facebook/wav2vec2-base-960h')
    _W2V_MODEL = Wav2Vec2Model.from_pretrained('facebook/wav2vec2-base-960h')
    _W2V_MODEL.eval()

proc = _W2V_PROC
w2v = _W2V_MODEL

# --- Helpers ---
frame_seconds = 0.5
sr = 16000
frame_len = int(frame_seconds * sr)


def build_chain_edge_index(n):
    edges = []
    for i in range(n - 1):
        edges.append([i, i + 1])
        edges.append([i + 1, i])
    return torch.tensor(edges, dtype=torch.long).t().contiguous() if len(edges) else torch.empty((2, 0), dtype=torch.long)


def estimate_breathing_rate_bpm(y, sr, audio_duration_s):
    if len(y) < sr:  # < 1s
        return None
    frame_len = int(0.2 * sr)
    hop_len = int(0.05 * sr)
    rms = librosa.feature.rms(y=y, frame_length=frame_len, hop_length=hop_len)[0]
    if len(rms) == 0:
        return None
    win = 5
    if len(rms) >= win:
        smooth = np.convolve(rms, np.ones(win) / win, mode='same')
    else:
        smooth = rms
    thr = float(smooth.mean() + 0.5 * smooth.std())
    times = librosa.frames_to_time(np.arange(len(smooth)), sr=sr, hop_length=hop_len)
    min_interval = 0.8  # seconds
    peaks = []
    last_t = -1e9
    for i in range(1, len(smooth) - 1):
        if smooth[i] > smooth[i - 1] and smooth[i] >= smooth[i + 1] and smooth[i] > thr:
            t = float(times[i])
            if t - last_t >= min_interval:
                peaks.append(t)
                last_t = t
    if len(peaks) < 2:
        return None
    bpm = (len(peaks) / max(audio_duration_s, 1e-6)) * 60.0
    return float(bpm)


def integrated_gradients_nodes(model, data, target='wheeze', steps=20):
    model.eval()
    x = data.x.detach()
    baseline = torch.zeros_like(x)
    grads_sum = torch.zeros_like(x)
    for alpha in np.linspace(0.0, 1.0, steps):
        x_scaled = (baseline + alpha * (x - baseline)).detach().requires_grad_(True)
        d = Data(x=x_scaled, edge_index=data.edge_index)
        w_logits, c_logits = model(d)
        out = w_logits if target == 'wheeze' else c_logits
        out_scalar = out.view(-1)[0]
        grad = torch.autograd.grad(out_scalar, x_scaled, retain_graph=False, create_graph=False)[0]
        grads_sum += grad
    avg_grads = grads_sum / float(steps)
    attributions = (x - baseline) * avg_grads
    node_attr = attributions.detach().cpu().norm(p=2, dim=1).numpy()
    return node_attr


def grad_times_input_nodes(model, data, target='wheeze'):
    model.eval()
    x = data.x.detach().clone().requires_grad_(True)
    d = Data(x=x, edge_index=data.edge_index)
    w_logits, c_logits = model(d)
    out = w_logits if target == 'wheeze' else c_logits
    out_scalar = out.view(-1)[0]
    grad = torch.autograd.grad(out_scalar, x, retain_graph=False, create_graph=False)[0]
    gxi = (grad * x).detach().cpu().norm(p=2, dim=1).numpy()
    return gxi


def topk_indices(arr, k):
    if arr is None or len(arr) == 0:
        return []
    k = min(int(k), len(arr))
    return [int(i) for i in np.argsort(arr)[-k:][::-1]]


def list_to_csv_value(v):
    if not v:
        return ''
    return ';'.join(str(x) for x in v)


# --- Batch inference per file ---
results = []
for audio_path in files:
    # Load raw audio
    y, _ = librosa.load(str(audio_path), sr=sr, mono=True)
    audio_duration_s = float(len(y) / sr)

    # Frame audio into 0.5s chunks
    frames = []
    for i in range(0, len(y), frame_len):
        f = y[i:i + frame_len]
        if len(f) < frame_len:
            f = np.pad(f, (0, frame_len - len(f)), mode='constant')
        frames.append(f.astype(np.float32))

    if len(frames) == 0:
        print('No frames extracted; skipping', audio_path)
        continue

    # Wav2Vec2 embeddings (batch)
    embeddings = []
    batch_size = 8
    with torch.no_grad():
        for i in range(0, len(frames), batch_size):
            batch = frames[i:i + batch_size]
            inputs = proc(batch, sampling_rate=sr, return_tensors='pt', padding=True)
            out = w2v(**inputs)
            emb = out.last_hidden_state.mean(dim=1).cpu().numpy().astype(np.float32)
            embeddings.append(emb)

    X = np.concatenate(embeddings, axis=0) if len(embeddings) else np.zeros((0, 768), dtype=np.float32)
    if X.shape[0] == 0:
        print('No embeddings computed; skipping', audio_path)
        continue

    # Create graph Data
    edge_index = build_chain_edge_index(X.shape[0])
    data = Data(x=torch.tensor(X, dtype=torch.float32), edge_index=edge_index)

    # Inference (full audio graph)
    start = time.time()
    with torch.no_grad():
        w_logits, c_logits = model(data)

    infer_ms = (time.time() - start) * 1000.0
    w_prob = float(torch.sigmoid(w_logits).view(-1)[0].cpu().item())
    c_prob = float(torch.sigmoid(c_logits).view(-1)[0].cpu().item())

    # Thresholds
    w_thr = 0.5
    c_thr = 0.5
    w_pred = int(w_prob >= w_thr)
    c_pred = int(c_prob >= c_thr)

    # Uncertainty/confidence
    w_unc = None
    c_unc = None
    w_conf = None
    c_conf = None
    if hasattr(model, 'log_var_wheeze'):
        w_logvar = float(model.log_var_wheeze.detach().cpu().item())
        w_unc = float(np.exp(0.5 * w_logvar))
        w_conf = float(1.0 / (1.0 + w_unc))
    if hasattr(model, 'log_var_crackle'):
        c_logvar = float(model.log_var_crackle.detach().cpu().item())
        c_unc = float(np.exp(0.5 * c_logvar))
        c_conf = float(1.0 / (1.0 + c_unc))

    # Breathing rate (full audio)
    breathing_rate_bpm = estimate_breathing_rate_bpm(y, sr, audio_duration_s)

    # Patient state manager (per file)
    state_mgr = PatientStateManager(ema_alpha=0.12, low_delta=0.08, high_delta=0.20, min_samples_for_baseline=5, force_established_after_s=10.0)
    patient_id = audio_path.stem

    # Cumulative windowed reasoning
    window_seconds = 5.0
    frames_per_window = max(1, int(round(window_seconds / frame_seconds)))
    window_rows = []

    for w_start in range(0, X.shape[0], frames_per_window):
        Xw = X[w_start:w_start + frames_per_window]
        if Xw.shape[0] == 0:
            continue
        ew = build_chain_edge_index(Xw.shape[0])
        dw = Data(x=torch.tensor(Xw, dtype=torch.float32), edge_index=ew)
        with torch.no_grad():
            w_l, c_l = model(dw)
        w_p = float(torch.sigmoid(w_l).view(-1)[0].cpu().item())
        c_p = float(torch.sigmoid(c_l).view(-1)[0].cpu().item())
        w_pd = int(w_p >= w_thr)
        c_pd = int(c_p >= c_thr)
        start_sec = float(w_start * frame_seconds)
        end_sec = float(min((w_start + Xw.shape[0]) * frame_seconds, audio_duration_s))

        # Window breathing rate
        start_sample = int(start_sec * sr)
        end_sample = int(end_sec * sr)
        y_window = y[start_sample:end_sample] if end_sample > start_sample else np.array([], dtype=np.float32)
        win_duration = max(end_sec - start_sec, 1e-6)
        breathing_rate_window = estimate_breathing_rate_bpm(y_window, sr, win_duration)

        # Patient state for this window
        state_out = state_mgr.update_and_get_state(patient_id, w_p, c_p, timestamp=start_sec)
        patient_state = state_out.get('overall_state')

        # Optional attributions per window
        ig_topk = []
        gxi_topk = []
        if compute_attributions:
            try:
                ig_scores = integrated_gradients_nodes(model, dw, target=attribution_target, steps=integrated_steps)
                gxi_scores = grad_times_input_nodes(model, dw, target=attribution_target)
                ig_topk = topk_indices(ig_scores, attr_topk)
                gxi_topk = topk_indices(gxi_scores, attr_topk)
            except Exception as e:
                print('Attribution failed for window', start_sec, '->', end_sec, 'error:', e)
                ig_topk = []
                gxi_topk = []

        window_rows.append({
            'start_sec': round(start_sec, 3),
            'end_sec': round(end_sec, 3),
            'num_frames': int(Xw.shape[0]),
            'wheeze_prob': round(w_p, 4),
            'crackle_prob': round(c_p, 4),
            'wheeze_pred': int(w_pd),
            'crackle_pred': int(c_pd),
            'patient_state': patient_state,
            'breathing_rate_bpm': None if breathing_rate_window is None else round(breathing_rate_window, 2),
            'ig_topk': ig_topk,
            'gxi_topk': gxi_topk,
        })

    if len(window_rows) > 0:
        w_win_probs = np.array([r['wheeze_prob'] for r in window_rows], dtype=np.float32)
        c_win_probs = np.array([r['crackle_prob'] for r in window_rows], dtype=np.float32)
        w_win_pred = np.array([r['wheeze_pred'] for r in window_rows], dtype=np.int32)
        c_win_pred = np.array([r['crackle_pred'] for r in window_rows], dtype=np.int32)
        win_summary = {
            'window_seconds': float(window_seconds),
            'frame_seconds': float(frame_seconds),
            'num_windows': int(len(window_rows)),
            'frames_per_window': int(frames_per_window),
            'wheeze_window_positive': int(w_win_pred.sum()),
            'crackle_window_positive': int(c_win_pred.sum()),
            'wheeze_window_ratio': float(w_win_pred.mean()),
            'crackle_window_ratio': float(c_win_pred.mean()),
            'wheeze_prob_mean_window': float(w_win_probs.mean()),
            'crackle_prob_mean_window': float(c_win_probs.mean()),
            'wheeze_prob_max_window': float(w_win_probs.max()),
            'crackle_prob_max_window': float(c_win_probs.max()),
        }
    else:
        win_summary = {
            'window_seconds': float(window_seconds),
            'frame_seconds': float(frame_seconds),
            'num_windows': 0,
            'frames_per_window': int(frames_per_window),
        }

    # Flags
    short_audio = audio_duration_s < frame_seconds
    near_threshold = (abs(w_prob - w_thr) <= 0.02) or (abs(c_prob - c_thr) <= 0.02)

    # Output payload
    request_id = uuid.uuid4().hex
    now_utc = datetime.datetime.now(datetime.timezone.utc)
    timestamp = now_utc.strftime('%Y-%m-%dT%H:%M:%SZ')

    output = {
        'request_id': request_id,
        'timestamp': timestamp,
        'result': {
            'audio_id': audio_path.stem,
            'audio_duration_s': round(audio_duration_s, 3),
            'wheeze': {
                'probability': round(w_prob, 4),
                'prediction': bool(w_pred),
                'confidence': None if w_conf is None else round(w_conf, 4),
            },
            'crackle': {
                'probability': round(c_prob, 4),
                'prediction': bool(c_pred),
                'confidence': None if c_conf is None else round(c_conf, 4),
            },
            'breathing_rate_bpm': None if breathing_rate_bpm is None else round(breathing_rate_bpm, 2),
            'model_version': ckpt.name,
            'inference_time_ms': round(float(infer_ms), 2),
        },
        'reasoning': {
            'thresholds': {'wheeze': w_thr, 'crackle': c_thr},
            'uncertainty_std': {'wheeze': w_unc, 'crackle': c_unc},
            'flags': {
                'short_audio': bool(short_audio),
                'near_threshold': bool(near_threshold),
            },
            'cumulative_windows': win_summary,
        }
    }

    # Save to gat_runs folder
    run_root = repo_root / 'gat_runs'
    run_id = now_utc.strftime('%Y%m%dT%H%M%SZ') + '_' + audio_path.stem + '_' + request_id[:8]
    run_dir = run_root / run_id
    run_dir.mkdir(parents=True, exist_ok=True)

    with open(run_dir / 'result.json', 'w', encoding='utf-8') as f:
        json.dump(output, f, indent=2)

    with open(run_dir / 'reasoning.json', 'w', encoding='utf-8') as f:
        json.dump(output['reasoning'], f, indent=2)

    # Save per-window details (JSON + CSV) for UI/debug
    window_report = {
        'audio_id': audio_path.stem,
        'audio_duration_s': round(audio_duration_s, 3),
        'window_seconds': float(window_seconds),
        'frame_seconds': float(frame_seconds),
        'attributions_enabled': bool(compute_attributions),
        'attribution_target': attribution_target if compute_attributions else None,
        'windows': window_rows,
    }

    with open(run_dir / 'window_report.json', 'w', encoding='utf-8') as f:
        json.dump(window_report, f, indent=2)

    with open(run_dir / 'window_report.csv', 'w', newline='', encoding='utf-8') as f:
        fieldnames = [
            'start_sec', 'end_sec', 'num_frames',
            'wheeze_prob', 'crackle_prob', 'wheeze_pred', 'crackle_pred',
            'patient_state', 'breathing_rate_bpm', 'ig_topk', 'gxi_topk'
        ]
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in window_rows:
            row_csv = dict(row)
            row_csv['ig_topk'] = list_to_csv_value(row_csv.get('ig_topk'))
            row_csv['gxi_topk'] = list_to_csv_value(row_csv.get('gxi_topk'))
            writer.writerow(row_csv)

    output['artifacts'] = {
        'result_json': 'result.json',
        'reasoning_json': 'reasoning.json',
        'window_report_json': 'window_report.json',
        'window_report_csv': 'window_report.csv',
        'run_dir': str(run_dir),
    }

    results.append(output)
    print('Saved outputs to:', run_dir)
    print(json.dumps(output, indent=2))

print('Completed', len(results), 'audio files.')


In [ ]:
# Inline color-based reasoning: auto-update window reports + combined outputs
import re
import pandas as pd
import statistics
from collections import Counter
from pathlib import Path

def safe_float(v):
    try:
        if v is None:
            return None
        return float(v)
    except Exception:
        return None

def summarize_run_dir(run_dir: Path):
    win_json = run_dir / 'window_report.json'
    win_csv = run_dir / 'window_report.csv'
    data = None
    windows = []
    audio_id = run_dir.name

    if win_json.exists():
        try:
            data = json.loads(win_json.read_text(encoding='utf-8'))
            windows = data.get('windows', [])
            audio_id = data.get('audio_id') or run_dir.name
        except Exception:
            return None
    elif win_csv.exists():
        try:
            df = pd.read_csv(win_csv)
            windows = df.to_dict(orient='records')
            audio_id = df['audio_id'].iloc[0] if 'audio_id' in df.columns else run_dir.name
        except Exception:
            return None
    else:
        return None

    if not windows:
        return None

    # collect numeric readings
    w_probs, c_probs, br_rates = [], [], []
    for w in windows:
        wp = safe_float(w.get('wheeze_prob') if isinstance(w, dict) else w.get('wheeze_prob'))
        cp = safe_float(w.get('crackle_prob') if isinstance(w, dict) else w.get('crackle_prob'))
        if wp is not None:
            w_probs.append(wp)
        if cp is not None:
            c_probs.append(cp)
        br = safe_float(w.get('breathing_rate_bpm'))
        if br is not None:
            br_rates.append(br)

    def mean_or_none(xs):
        try:
            return float(statistics.mean(xs)) if xs else None
        except Exception:
            return None

    w_mean = mean_or_none(w_probs)
    c_mean = mean_or_none(c_probs)
    br_mean = mean_or_none(br_rates)
    br_min = min(br_rates) if br_rates else None
    br_max = max(br_rates) if br_rates else None

    # per-window comment & color (first 10s forced to establishing/grey)
    state_color_map = {'green': 'green', 'orange': 'orange', 'red': 'red', 'establishing': 'grey'}
    processed_states = []
    for idx, w in enumerate(windows):
        pstate = w.get('patient_state') if isinstance(w, dict) else w.get('patient_state', '')
        pstate = (str(pstate).lower().strip() if pstate is not None else 'establishing')
        start_sec = safe_float(w.get('start_sec') if isinstance(w, dict) else w.get('start_sec'))
        if start_sec is None:
            start_sec = float(idx * 5.0)

        if start_sec < 10.0:
            final_state = 'establishing'
        else:
            final_state = pstate if pstate in ('green', 'orange', 'red', 'establishing') else 'establishing'

        state_color = state_color_map.get(final_state, 'grey')
        if final_state == 'red':
            comment = 'RED — patient requires clinical review.'
        elif final_state == 'orange':
            comment = 'ORANGE — nurse should be cautious.'
        elif final_state == 'green':
            comment = 'GREEN — no immediate attention required.'
        else:
            comment = 'GREY — establishing baseline — insufficient history; interpret with caution.'

        try:
            w['state_color'] = state_color
            w['comment'] = comment
        except Exception:
            pass

        processed_states.append(final_state)

    counts = Counter(processed_states)
    overall = 'establishing'
    for s in ['red', 'orange', 'green', 'establishing']:
        if counts.get(s, 0) > 0:
            overall = s
            break

    latest_comment = windows[-1].get('comment') if windows else ''
    combined = latest_comment

    extras = []
    if w_mean is not None:
        extras.append(f'mean wheeze prob {w_mean:.3f}')
    if c_mean is not None:
        extras.append(f'mean crackle prob {c_mean:.3f}')
    if br_mean is not None:
        extras.append(f'mean breathing rate {br_mean:.1f} bpm (min {br_min:.1f}, max {br_max:.1f})')

    if extras:
        combined = combined + ' ' + '; '.join(extras) + '.'

    # persist updated window report (JSON + CSV)
    try:
        if win_json.exists() and data is not None:
            data['windows'] = windows
            win_json.write_text(json.dumps(data, indent=2, ensure_ascii=False), encoding='utf-8')
    except Exception:
        pass

    try:
        if win_csv.exists():
            df_out = pd.DataFrame(windows)
            for c in ['state_color', 'comment']:
                if c not in df_out.columns:
                    df_out[c] = ''
            for col in ['ig_topk', 'gxi_topk']:
                if col in df_out.columns:
                    df_out[col] = df_out[col].apply(lambda v: ';'.join(v) if isinstance(v, (list, tuple)) else ('' if pd.isna(v) else str(v)))
            df_out.to_csv(win_csv, index=False)
    except Exception:
        pass

    return {
        'run_dir': str(run_dir),
        'audio_id': str(audio_id),
        'num_windows': int(len(windows)),
        'counts': dict(counts),
        'mean_wheeze_prob': w_mean,
        'mean_crackle_prob': c_mean,
        'breathing_rate_mean': br_mean,
        'breathing_rate_min': br_min,
        'breathing_rate_max': br_max,
        'overall_state': overall,
        'latest_window_comment': latest_comment,
        'latest_state_color': (windows[-1].get('state_color') if windows else None),
        'combined_reasoning': combined,
    }



In [ ]:
# generate_combined_reasoning removed — color_reasoning is canonical
print('generate_combined_reasoning removed; use *_color_reasoning.json/.txt outputs only')
